# Personalized Meta-RL Strategy Selector — `06_meta_rl_selector.ipynb`

Sixth and final notebook of the **Personalized AI Trading Agent** research project. A **PPO** agent learns *which of the five pre-trained, frozen specialised policies should control the next decision*. It never predicts BUY / SELL / HOLD / EXIT itself.

| Id | Policy | Artifact (from notebooks 01–05) |
|---|---|---|
| 0 | Momentum | `momentum_policy.pt` |
| 1 | Mean-Reversion | `mean_reversion_policy.pt` |
| 2 | Trend-Following | `trend_following_policy.pt` |
| 3 | Breakout | `breakout_policy.pt` |
| 4 | Conservative Risk-Aware | `conservative_policy.pt` |

**One decision step:** observation (market, investor, position, policy proposals + their past performance) → PPO selects a policy → that **frozen** policy proposes an action → a **separate deterministic risk controller** validates it (and can reject it or force an exit, whatever PPO selected) → execution at the **next bar's open** with fees and slippage → reward → time advances without exposing future data.

> **Research prototype for offline evaluation and paper trading only.** Nothing here is a trading recommendation, and no simulated result implies live profitability. The notebook reports whether the selector actually beats its baselines on held-out data; it makes no claim of superiority unless the results support it.

### Honest limitations that shape the design
* **No real investor history is supplied**, so investors are **SYNTHETIC profiles** generated by documented rules (section 6). The selector can therefore *not* be said to have learned any real individual's behaviour. A function to compute behaviour statistics from a real trade history is included for future use.
* **No policy is ever substituted.** If an artifact is missing the notebook stops and tells you what to upload; there are no random or fake weights anywhere.
* **Leakage control on the policy side:** the five policies were trained on data up to their `train_end` and validated up to their `val_end`. By default (`policy_oos`) the selector is trained and validated only on the data **after** the policies' training period (the policies' own validation window, split in two) and tested on the policies' held-out test period, so the selector never learns from the policies' in-sample behaviour. This leaves a *short* training period, so results are noisy; the `full_history` mode exists but is flagged as in-sample for the policies.
* The test period is the same held-out period as in notebooks 01–05. Do not retune policies after looking at it.

### Notebook map
1 Install and configuration · 2 Artifact upload and validation · 3 Modules (risk controller, environment core) · 4 Data · 5 Policy loading and compatibility checks · 6 Investors, observation schema, market data and shadow performance · 7 Risk-controller tests and environment checks · 8 PPO training · 9 Validation · 10 Held-out test, baselines, ablations, reward sensitivity · 11 Analysis (regimes, investor profiles, selection, rejections, behavioural consistency) · 12 Plots · 13 Export · 14 Inference demonstration · 15 Integration report

## 1. Install
Colab already ships PyTorch, pandas, NumPy, scikit-learn and Matplotlib. `gymnasium`, `stable-baselines3` and `yfinance` are installed or upgraded. All indicators are implemented manually (no `pandas-ta`, which is fragile on new NumPy/Python versions).

In [ ]:
import subprocess, sys

def _pip_install(*packages):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])
    except Exception as exc:  # noqa: BLE001 - installation problems must not hide the real error later
        print(f"WARNING: pip install {packages} failed: {exc!r}. Continuing with the installed versions, if any.")

_pip_install("yfinance", "gymnasium", "stable-baselines3")

## 1. Configuration
Everything configurable lives in one `CONFIG`: data, artifact location, split mode, execution costs and capital, reward weights, risk limits, investor options, PPO hyper-parameters and training budget, ablations and reward-sensitivity variants.

* **Artifacts:** `source = "upload"` (Colab file upload of the five zips from notebooks 01–05), `"drive"` (a Google Drive folder) or `"path"` (a folder that already contains them).
* **Splits:** `policy_oos` (default, see the limitations above) or `full_history`. `meta_val_start=None` places the meta-validation start 75% into the policies' validation period.
* **Reward:** all terms are expressed in units of a reference daily volatility (`sigma_ref`); see section 6 for the formula and the rejection rule.
* **Training budget:** `total_timesteps` per PPO run with validation-based early stopping (`patience_evals`). Ablations and reward-sensitivity runs use the same budget so comparisons are fair.

In [ ]:
import os, json, math, time, random, pickle, warnings, platform, importlib, shutil, copy, hashlib, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
IN_COLAB = Path("/content").exists()

CONFIG = {
    "seed": 42,
    # ---- data (use the SAME symbols/benchmark/dates as notebooks 01-05 unless you know why not) --------
    "symbols": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "XOM", "CVX", "JNJ", "PG", "KO", "PEP", "WMT", "V", "HD"],
    "benchmark": "SPY",
    "start_date": "2010-01-01",
    "end_date": "2025-12-31",          # exclusive
    "min_history_rows": 800,
    "local_csv_dir": None,             # optional fallback: <dir>/<SYMBOL>.csv with Date,Open,High,Low,Close,Volume (real data only)
    # ---- the five policy artifacts -------------------------------------------------------------------
    "artifacts": {"source": "upload",  # "upload" | "drive" | "path"
                  "root": "/content/sp_policy_artifacts" if IN_COLAB else "./sp_policy_artifacts",
                  "drive_dir": "/content/drive/MyDrive/trading_agent/policy_artifacts"},
    # ---- chronological splits for the SELECTOR ---------------------------------------------------------
    "splits": {"mode": "policy_oos", "meta_val_start": None},
    # ---- execution (identical for the selector and every baseline) -----------------------------------
    "execution": {"initial_capital": 100000.0, "fee_bps": 5.0, "slippage_bps": 5.0, "short_borrow_bps_annual": 50.0, "min_trade_fraction": 0.01,
                  "dd_lookback": 120},   # drawdown is measured below the trailing 120-bar equity high, so a halt expires after a cooling-off period
    # ---- reward (see section 6) -------------------------------------------------------------------------
    "reward": {"w_perf": 1.0, "w_drawdown": 1.0, "w_cost": 0.5, "w_turnover": 0.5, "w_behavior": 0.1, "sigma_ref": 0.01,
               "dd_tolerance_fraction": 0.5, "turnover_charge": 0.001, "rejection_penalty": 0.2, "forced_exit_penalty": 0.0, "rf_annual": 0.0},
    # ---- deterministic risk controller (global limits; investor-specific limits come from the investor profile) ---
    "risk_limits": {"max_position_fraction": 0.60, "max_portfolio_exposure": 0.60, "position_stop_loss": 0.06, "max_holding_bars": 120,
                    "max_daily_loss": 0.03, "max_recent_loss": 0.06, "drawdown_halt_fraction": 0.60, "max_vol_percentile": 0.90,
                    "max_abs_gap_atr": 3.0, "max_abnormal_ret_z": 4.0, "market_stress_dd": 0.12},
    "investor": {"history_csv": None},  # optional REAL trade history (see section 6); None -> SYNTHETIC investors
    "env": {"episode_length": 120, "n_envs": 16},
    # ---- PPO ----------------------------------------------------------------------------------------------
    "ppo": {"total_timesteps": 80_000, "n_steps": 120, "batch_size": 240, "n_epochs": 10, "learning_rate": 3e-4, "gamma": 0.99,
            "gae_lambda": 0.95, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "max_grad_norm": 0.5, "net_arch": [128, 128],
            "device": "auto", "eval_every_rollouts": 3, "patience_evals": 8, "min_evals": 8},
    "ablations": {"run": True},
    "reward_sensitivity": {"run": True, "variants": {
        "no_behavior_term": {"w_behavior": 0.0},
        "strong_drawdown_penalty": {"w_drawdown": 3.0},
        "strong_cost_and_turnover_penalty": {"w_cost": 2.0, "w_turnover": 2.0}}},
    "evaluation": {"archetypes": ["conservative", "balanced", "aggressive"], "bootstrap_reps": 1000, "bootstrap_block": 10},
    "export_dir": "/content/sp06_meta_rl_artifacts" if IN_COLAB else "./sp06_meta_rl_artifacts",
}

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
ART_DIR = Path(CONFIG["export_dir"]); ART_DIR.mkdir(parents=True, exist_ok=True)
POLICY_ROOT = Path(CONFIG["artifacts"]["root"]); POLICY_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__} | PPO device setting: {CONFIG['ppo']['device']} (available: {DEVICE_NAME})")
print("The five frozen policies and the environment run on CPU (tiny networks, batch size 1..N); PPO's own MLP can use the GPU or the CPU.")
print(f"export dir: {ART_DIR.resolve()}")

## 2. Artifact upload and validation
Each of notebooks 01–05 produced a zip (for example `sp01_momentum_artifacts.zip`) containing the `.pt` file, its config, `feature_schema.json`, `preprocessing.pkl` and the exact `sp0X_features.py` / `sp0X_model.py` code. Because every zip contains files with the **same names** (`feature_schema.json`, `preprocessing.pkl`), **each policy needs its own folder**. This cell:

* accepts the zips via Colab upload, a Google Drive folder, or a local folder that already holds them, and unzips each into its own folder;
* verifies that **all five** artifact sets are present with every required file;
* **stops with a clear error** if anything is missing. There is no fallback, no random model and no fake weights.

In [ ]:
REQUIRED = {
    "momentum": ["momentum_policy.pt", "momentum_config.json", "feature_schema.json", "preprocessing.pkl", "sp01_features.py", "sp01_model.py"],
    "mean_reversion": ["mean_reversion_policy.pt", "mean_reversion_config.json", "feature_schema.json", "preprocessing.pkl", "sp02_features.py", "sp02_model.py"],
    "trend_following": ["trend_following_policy.pt", "trend_following_config.json", "feature_schema.json", "preprocessing.pkl", "sp03_features.py", "sp03_model.py"],
    "breakout": ["breakout_policy.pt", "breakout_config.json", "feature_schema.json", "preprocessing.pkl", "sp04_features.py", "sp04_model.py"],
    "conservative": ["conservative_policy.pt", "conservative_config.json", "feature_schema.json", "preprocessing.pkl", "sp05_features.py", "sp05_model.py"],
}
SOURCE_NOTEBOOK = {"momentum": "01_momentum_policy.ipynb", "mean_reversion": "02_mean_reversion_policy.ipynb", "trend_following": "03_trend_following_policy.ipynb",
                   "breakout": "04_breakout_policy.ipynb", "conservative": "05_conservative_policy.ipynb"}

def _unzip_all(root: Path) -> None:
    for z in sorted(root.glob("*.zip")):
        target = root / z.stem
        if not target.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
            print(f"unzipped {z.name} -> {target.name}/")

def _find_folders(root: Path) -> dict:
    found = {}
    for key, files in REQUIRED.items():
        hits = [p.parent for p in root.rglob(files[0])]
        if hits:
            found[key] = hits[0]
    return found

def _missing_report(root: Path) -> list:
    problems = []
    found = _find_folders(root)
    for key, files in REQUIRED.items():
        if key not in found:
            problems.append(f"  - {key}: '{files[0]}' not found. Run {SOURCE_NOTEBOOK[key]} and upload the zip it produced (e.g. {Path(files[4]).stem.split('_')[0]}_*_artifacts.zip).")
        else:
            lacking = [f for f in files if not (found[key] / f).exists()]
            if lacking:
                problems.append(f"  - {key}: folder {found[key]} is missing {lacking}")
    return problems

src = CONFIG["artifacts"]["source"]
if src == "drive":
    from google.colab import drive                                   # only available in Colab
    drive.mount("/content/drive")
    for f in Path(CONFIG["artifacts"]["drive_dir"]).glob("*"):
        if f.suffix == ".zip" or f.is_dir():
            dest = POLICY_ROOT / f.name
            if f.is_dir() and not dest.exists():
                shutil.copytree(f, dest)
            elif f.suffix == ".zip" and not dest.exists():
                shutil.copy(f, dest)
elif src == "upload" and _missing_report(POLICY_ROOT):
    try:
        from google.colab import files                                # only available in Colab
        print("Please upload the five artifact zips produced by notebooks 01-05 (select all five files)...")
        for name, data in files.upload().items():
            (POLICY_ROOT / name).write_bytes(data)
    except ImportError:
        print("Not running in Colab: place the five zips (or their unzipped folders) under", POLICY_ROOT.resolve())
_unzip_all(POLICY_ROOT)

problems = _missing_report(POLICY_ROOT)
if problems:
    raise FileNotFoundError(
        "Cannot continue: required policy artifacts are missing. The selector is never trained on substitute or random models.\n"
        + "\n".join(problems)
        + f"\nUpload the missing files (Colab: Files panel -> upload, or set CONFIG['artifacts']['source']='drive'/'path') into {POLICY_ROOT.resolve()} and re-run this cell.")
POLICY_FOLDERS = _find_folders(POLICY_ROOT)
print("\nAll five artifact sets found:")
for key, folder in POLICY_FOLDERS.items():
    print(f"  {key:16s} -> {folder}")

## 3. Modules: deterministic risk controller and environment core
Two small modules are written to disk and imported. They ship with the selector so a backend can use exactly the same code.

* `meta_rl_risk.py` — the **deterministic risk controller** (`MetaRiskController`). No neural network, no learned parameter. Configurable: maximum position size, maximum portfolio exposure, maximum daily loss, maximum drawdown (the investor's declared limit), volatility restrictions, invalid-action checks and authorisation checks (kill switch, short authorisation). It can reject any proposal regardless of PPO's selection or confidence, always allows de-risking exits, and reports a reason for every rejection or override.
* `meta_rl_core.py` — policy bank (loads the five frozen policies), aligned market data with each policy's own past-only features, investors, account/execution accounting, the reward function, the observation schema, the Gymnasium environments, the batch evaluator and the inference interface (`load_meta_agent`, `select_policy`, `predict_action`, `save_selection_log`).

Selection (PPO), policy inference (frozen policies), risk validation (`MetaRiskController`) and simulated execution (`AccountBatch`) are separate classes/functions.

In [ ]:
%%writefile meta_rl_risk.py
"""Deterministic risk controller for the Personalized Meta-RL Strategy Selector.

This module contains NO neural network and NO learned parameters. Whatever policy the PPO selector picks, and however
confident anyone is, every proposed action passes through ``MetaRiskController`` before simulated or paper execution.
The controller can reject an entry or force an exit; it never lets the selector bypass it. De-risking proposals
(EXIT) are always allowed, even when trading is not authorised.

Checks (each rejection / override is reported with its reason):
  invalid_action          proposal impossible for the current position (e.g. BUY while long, EXIT while flat) -> HOLD
  trading_not_authorised  kill switch: no new entries (EXIT still allowed)
  short_not_authorised    shorting not authorised for this investor
  position_stop_loss      unrealised loss reached the hard stop                -> forced EXIT
  max_holding_period      holding period reached the hard limit                -> forced EXIT
  drawdown_flatten        account drawdown reached the investor's limit        -> forced EXIT
  drawdown_halt           account drawdown reached the halt fraction of the limit -> entries blocked
  daily_loss_halt         the last bar's loss reached the daily loss limit     -> entries blocked
  recent_loss_halt        ~20-bar equity loss reached its limit                -> entries blocked
  volatility_restriction  volatility percentile above its cap                  -> entries blocked
  abnormal_gap_or_move    abnormal gap / move                                  -> entries blocked
  market_stress           benchmark far below its recent high                  -> entries blocked
  exposure_limit          position or portfolio exposure limit would be exceeded -> entries blocked
  risk_data_missing       risk inputs unavailable (fail safe)                  -> entries blocked
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np

HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTION_NAMES = ["HOLD", "BUY", "SELL", "EXIT"]

REASON_NAMES = [
    "invalid_action", "trading_not_authorised", "short_not_authorised", "position_stop_loss", "max_holding_period",
    "drawdown_flatten", "drawdown_halt", "daily_loss_halt", "recent_loss_halt", "volatility_restriction",
    "abnormal_gap_or_move", "market_stress", "exposure_limit", "risk_data_missing",
]
FORCED_EXIT_REASONS = ["position_stop_loss", "max_holding_period", "drawdown_flatten"]
BLOCK_REASONS = [r for r in REASON_NAMES if r not in FORCED_EXIT_REASONS and r != "invalid_action"]
REQUIRED_RISK_FEATURES = ["atr_pct_pctile_252", "rv20_pctile_252", "gap_abs_atr", "abnormal_ret_z", "bench_dd_60"]


def valid_action_mask(position, allow_short=True) -> np.ndarray:
    """flat -> HOLD/BUY/SELL(if short allowed); long or short -> HOLD/EXIT."""
    pos = np.asarray(position).astype(int).reshape(-1)
    allow = np.broadcast_to(np.asarray(allow_short, dtype=bool), pos.shape)
    m = np.zeros((pos.size, 4), dtype=bool)
    m[:, HOLD] = True
    flat = pos == 0
    m[flat, BUY] = True
    m[:, SELL] = flat & allow
    m[~flat, EXIT] = True
    return m


@dataclass
class MetaRiskLimits:
    """Global limits. Investor-specific limits (drawdown limit, maximum position fraction, short authorisation,
    trading authorisation) are passed per call as arrays (see ``review_batch``)."""
    max_position_fraction: float = 0.60       # hard cap on one position as a fraction of account equity
    max_portfolio_exposure: float = 0.60      # gross exposure / equity after an entry
    position_stop_loss: float = 0.06          # hard per-position loss -> forced EXIT
    max_holding_bars: int = 120               # hard time limit -> forced EXIT (0 disables)
    max_daily_loss: float = 0.03              # a bar loss at least this large blocks new entries
    max_recent_loss: float = 0.06             # ~20-bar equity loss that blocks new entries
    drawdown_halt_fraction: float = 0.60      # entries blocked once drawdown >= fraction * investor drawdown limit
    max_vol_percentile: float = 0.90          # volatility restriction (percentile of ATR% / realised vol)
    max_abs_gap_atr: float = 3.0
    max_abnormal_ret_z: float = 4.0
    market_stress_dd: float = 0.12

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "MetaRiskLimits":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


class MetaRiskController:
    """Final authority over execution. Pure, deterministic, vectorised over B sleeves."""

    def __init__(self, limits: MetaRiskLimits | None = None):
        self.limits = limits or MetaRiskLimits()

    def entry_fraction(self, desired_fraction, investor_max_fraction):
        """Largest entry size allowed (fraction of equity): min(desired, investor cap, global cap)."""
        return np.minimum(np.minimum(desired_fraction, investor_max_fraction), self.limits.max_position_fraction)

    def review_batch(self, proposed, position, unrealized_pnl, holding_bars, features: dict, account: dict, investor: dict):
        """Review one proposal per sleeve.

        proposed, position, unrealized_pnl, holding_bars : arrays (B,)
        features : RAW latest-bar arrays (B,) containing REQUIRED_RISK_FEATURES
        account  : arrays/scalars  drawdown (>=0), gross_exposure (current), daily_return (last bar), recent_loss (>=0),
                   entry_fraction (planned size of a new entry)
        investor : arrays/scalars  drawdown_limit, max_position_fraction, allow_short, authorised
        Returns (final_actions (B,), reasons (B, len(REASON_NAMES)) bool).
        """
        lim = self.limits
        proposed = np.asarray(proposed).astype(int).reshape(-1)
        n = proposed.size
        f = lambda x: np.broadcast_to(np.asarray(x, dtype=float), (n,))
        pos = np.asarray(position).astype(int).reshape(-1)
        pnl, hold = f(unrealized_pnl), f(holding_bars)
        dd, gross, dret, rloss = f(account["drawdown"]), f(account["gross_exposure"]), f(account["daily_return"]), f(account["recent_loss"])
        frac = f(account["entry_fraction"])
        dd_lim = f(investor["drawdown_limit"])
        allow_short = np.broadcast_to(np.asarray(investor["allow_short"], dtype=bool), (n,))
        authorised = np.broadcast_to(np.asarray(investor["authorised"], dtype=bool), (n,))
        inv_cap = f(investor["max_position_fraction"])
        R = {k: np.zeros(n, dtype=bool) for k in REASON_NAMES}
        final = proposed.copy()

        valid = valid_action_mask(pos, True)
        bad = ~valid[np.arange(n), final]
        R["short_not_authorised"] = (final == SELL) & (pos == 0) & ~allow_short
        R["invalid_action"] = bad
        final[bad | R["short_not_authorised"]] = HOLD

        entry = (final == BUY) | (final == SELL)
        R["trading_not_authorised"] = entry & ~authorised
        with np.errstate(invalid="ignore"):
            feats = {k: np.asarray(features[k], dtype=float).reshape(-1) for k in REQUIRED_RISK_FEATURES}
            vol_pct = np.maximum(feats["atr_pct_pctile_252"], feats["rv20_pctile_252"])
            known = np.all([np.isfinite(v) for v in feats.values()], axis=0)
            R["risk_data_missing"] = entry & ~known
            R["drawdown_halt"] = entry & (dd >= lim.drawdown_halt_fraction * dd_lim)
            R["daily_loss_halt"] = entry & (dret <= -lim.max_daily_loss)
            R["recent_loss_halt"] = entry & (rloss >= lim.max_recent_loss)
            R["volatility_restriction"] = entry & (vol_pct > lim.max_vol_percentile)
            R["abnormal_gap_or_move"] = entry & ((feats["gap_abs_atr"] > lim.max_abs_gap_atr) | (np.abs(feats["abnormal_ret_z"]) > lim.max_abnormal_ret_z))
            R["market_stress"] = entry & (feats["bench_dd_60"] <= -lim.market_stress_dd)
        cap = np.minimum(inv_cap, lim.max_position_fraction)
        R["exposure_limit"] = entry & ((frac > cap + 1e-12) | (gross + frac > lim.max_portfolio_exposure + 1e-12))
        blocked = np.zeros(n, dtype=bool)
        for k in BLOCK_REASONS:
            blocked |= R[k]
        final[blocked] = HOLD

        held = pos != 0
        with np.errstate(invalid="ignore"):
            R["position_stop_loss"] = held & (pnl <= -lim.position_stop_loss)
            R["max_holding_period"] = held & (lim.max_holding_bars > 0) & (hold >= lim.max_holding_bars)
        R["drawdown_flatten"] = held & (dd >= dd_lim)
        forced = R["position_stop_loss"] | R["max_holding_period"] | R["drawdown_flatten"]
        final[forced] = EXIT
        reasons = np.stack([R[k] for k in REASON_NAMES], axis=1)
        return final, reasons

    def review_one(self, proposed_action, position, unrealized_pnl, holding_bars, features: dict, account: dict, investor: dict) -> dict:
        """Single-state convenience wrapper with a readable result."""
        wrap = lambda d: {k: np.asarray([v]) if np.ndim(v) == 0 else np.asarray(v) for k, v in d.items()}
        final, reasons = self.review_batch([proposed_action], [position], [unrealized_pnl], [holding_bars],
                                           wrap(features), wrap(account), wrap(investor))
        names = [r for r, on in zip(REASON_NAMES, reasons[0]) if on]
        return {"proposed_action": int(proposed_action), "proposed_action_name": ACTION_NAMES[int(proposed_action)],
                "final_action": int(final[0]), "final_action_name": ACTION_NAMES[int(final[0])],
                "approved": bool(final[0] == proposed_action), "reasons": names}


In [ ]:
%%writefile meta_rl_core.py
"""Core of the Personalized Meta-RL Strategy Selector (offline evaluation / paper trading research prototype).

Contents
  PolicyBank        loads the five FROZEN specialised policies from their saved artifacts (no substitutes)
  MarketData        aligned OHLCV + per-policy features (past-only) + selector market context
  Investor          synthetic (or user-supplied) investor profile, behaviour history and risk limits
  AccountBatch      vectorised single-asset sleeves: cash, shares, fees, slippage, next-open execution
  SleeveSim         one decision step: observation -> proposals -> selection -> deterministic risk gate -> execution -> reward
  run_shadow        past-only simulated performance of every policy (candidate-policy information)
  MetaTradingEnv    Gymnasium environment (B = 1) used by Stable-Baselines3 PPO
  run_selector      batch evaluation of any selector over a chronological segment
  MetaAgent         inference interface: select_policy / predict_action / save_selection_log

Selection, policy inference, risk validation and simulated execution are separate functions/classes.
Nothing in this module downloads data or trains anything.
"""
from __future__ import annotations

import importlib
import json
import sys
from dataclasses import asdict, dataclass, field
from pathlib import Path

import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from gymnasium import spaces

from meta_rl_risk import (ACTION_NAMES, BUY, EXIT, HOLD, REASON_NAMES, REQUIRED_RISK_FEATURES, SELL,
                          FORCED_EXIT_REASONS, MetaRiskController, MetaRiskLimits, valid_action_mask)

POLICY_KEYS = ["momentum", "mean_reversion", "trend_following", "breakout", "conservative"]
POLICY_NAMES = ["Momentum Policy", "Mean-Reversion Policy", "Trend-Following Policy", "Breakout Policy", "Conservative Risk-Aware Policy"]
N_POLICIES = 5
POLICY_SPEC = {
    "momentum": dict(pid="SP-01", pt="momentum_policy.pt", cfg="momentum_config.json", feat="sp01_features", model="sp01_model", cls="MomentumPolicy"),
    "mean_reversion": dict(pid="SP-02", pt="mean_reversion_policy.pt", cfg="mean_reversion_config.json", feat="sp02_features", model="sp02_model", cls="MeanReversionPolicy"),
    "trend_following": dict(pid="SP-03", pt="trend_following_policy.pt", cfg="trend_following_config.json", feat="sp03_features", model="sp03_model", cls="TrendFollowingPolicy"),
    "breakout": dict(pid="SP-04", pt="breakout_policy.pt", cfg="breakout_config.json", feat="sp04_features", model="sp04_model", cls="BreakoutPolicy"),
    "conservative": dict(pid="SP-05", pt="conservative_policy.pt", cfg="conservative_config.json", feat="sp05_features", model="sp05_model", cls="ConservativePolicy"),
}
COMMON_FILES = ["feature_schema.json", "preprocessing.pkl"]
EXPECTED_ACTIONS = {"0": "HOLD", "1": "BUY", "2": "SELL", "3": "EXIT"}
REGIME_NAMES = ["sideways", "uptrend", "up_weakening", "downtrend", "down_weakening"]


def required_files(key: str) -> list:
    s = POLICY_SPEC[key]
    return [s["pt"], s["cfg"], *COMMON_FILES, s["feat"] + ".py", s["model"] + ".py"]


# =====================================================================================================
# Policy bank (frozen policies)
# =====================================================================================================
def _import_fresh(folder: Path, name: str):
    """Import module `name` from `folder`, discarding any stale module of the same name from another location."""
    if name in sys.modules:
        f = getattr(sys.modules[name], "__file__", "") or ""
        if Path(f).resolve().parent != folder.resolve():
            del sys.modules[name]
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
    importlib.invalidate_caches()
    return importlib.import_module(name)


class PolicyBank:
    """The five frozen specialised policies. Their weights are never updated by anything in this notebook."""

    def __init__(self, folders: dict, device: str = "cpu"):
        self.folders = {k: Path(v) for k, v in folders.items()}
        self.device = torch.device(device)
        self.feat, self.policies, self.static_fn = [], [], []
        for key in POLICY_KEYS:
            spec, folder = POLICY_SPEC[key], self.folders[key]
            fm = _import_fresh(folder, spec["feat"])
            mm = _import_fresh(folder, spec["model"])
            pol = getattr(mm, spec["cls"]).load(folder / spec["pt"], self.device)
            for prm in pol.net.parameters():
                prm.requires_grad_(False)                        # frozen
            pol.net.eval()
            self.feat.append(fm); self.policies.append(pol)
            self.static_fn.append(self._static_builder(key, fm))
        self.seq_len = self.policies[0].seq_len
        self.columns = [list(fm.FEATURE_COLUMNS) for fm in self.feat]

    @staticmethod
    def _static_builder(key, fm):
        if key == "momentum":
            return lambda c, prof: fm.encode_static(c["pos"], prof)
        if key == "conservative":
            return lambda c, prof: fm.encode_static(c["pos"], prof, c["pnl"], c["hold"], c["pdd"], c["gross"], c["size"], c["rloss"])
        return lambda c, prof: fm.encode_static(c["pos"], prof, c["pnl"], c["hold"])

    def proposals(self, md, k: int, sym: np.ndarray, ctx: dict, prof: np.ndarray, which=range(N_POLICIES)):
        """Calibrated action probabilities of the requested policies for B sleeves at decision index k -> (B, P, 4)."""
        B = len(sym)
        probs = np.zeros((B, N_POLICIES, 4), dtype=np.float32)
        kk = np.broadcast_to(np.asarray(k), (B,))
        idx = kk[:, None] + np.arange(-self.seq_len + 1, 1)[None, :]                      # (B, L) window rows, all <= decision bar
        for p in which:
            pol = self.policies[p]
            seq = md.xs[p][sym[:, None], idx]                                             # (B, L, F_p) pre-scaled, past-only
            static = self.static_fn[p](ctx, prof)
            valid = valid_action_mask(ctx["pos"], pol.allow_short)
            with torch.no_grad():
                pr = pol.predict_scaled(torch.from_numpy(seq).to(pol.device), torch.from_numpy(static).to(pol.device),
                                        torch.from_numpy(valid).to(pol.device))
            probs[:, p] = pr.cpu().numpy()
        return probs


# =====================================================================================================
# Market data (aligned, past-only features)
# =====================================================================================================
MARKET_FEATURES = [
    "ret_5", "ret_20", "ret_60", "ema20_ema50", "close_sma200", "sma50_sma200", "adx_14", "di_diff", "atr_pct", "rv_20", "rsi_14",
    "vol_ratio_20", "vol_z_60", "rel_strength_20", "rel_strength_60", "bench_ret_20", "bench_rv_20",                 # shared base (17)
    "efficiency_ratio_20", "trend_r2_60",                                                                           # trend persistence (SP-03 features)
    "dd_from_high_60", "atr_pct_pctile_252", "rv20_pctile_252", "bench_dd_60", "bench_rv20_pctile_252",              # risk / volatility context (SP-05 features)
    "zscore_20", "bb_pos_20", "brk_dist_high_20", "brk_dist_low_20", "brk_vol_surge_20",                            # range / breakout context (SP-02 / SP-04 features)
]
N_MARKET = len(MARKET_FEATURES) + len(REGIME_NAMES)


class MarketData:
    """Aligned daily OHLCV of S symbols plus, for each policy, its own past-only feature matrix (raw and pre-scaled)."""

    def __init__(self, raw_data: dict, bench_df: pd.DataFrame, bank: PolicyBank):
        self.symbols = list(raw_data)
        cal = bench_df.index
        for s in self.symbols:
            cal = cal.intersection(raw_data[s].index)
        self.cal = cal.sort_values()
        self.T, self.S = len(self.cal), len(self.symbols)
        if self.T < 600:
            raise ValueError(f"only {self.T} common dates across symbols and benchmark; need at least 600")
        frames = {s: raw_data[s].loc[self.cal] for s in self.symbols}
        self.bench = bench_df.loc[self.cal]
        self.open = np.stack([frames[s]["Open"].to_numpy(float) for s in self.symbols], axis=1)      # (T, S)
        self.close = np.stack([frames[s]["Close"].to_numpy(float) for s in self.symbols], axis=1)
        self.bench_close = self.bench["Close"].to_numpy(float)
        self.bank = bank
        self.raw, self.xs = [], []
        for p in range(N_POLICIES):
            fm, pol = bank.feat[p], bank.policies[p]
            arr = np.stack([fm.build_features(frames[s], self.bench)[list(fm.FEATURE_COLUMNS)].ffill(limit=3).to_numpy(np.float32)
                            for s in self.symbols], axis=0)                                           # (S, T, F_p)
            self.raw.append(arr)
            self.xs.append(pol.scale(arr))                                                           # same scaling as in training
        self.name2loc = {}
        for p in range(N_POLICIES):
            for j, n in enumerate(bank.columns[p]):
                self.name2loc.setdefault(n, (p, j))
        L = bank.seq_len
        firsts = [int(np.argmax(np.isfinite(self.raw[p]).all(axis=2), axis=1).max()) for p in range(N_POLICIES)]
        self.valid_start = max(firsts) + L
        self.mkt_raw = np.stack([self.get(n).T for n in MARKET_FEATURES], axis=2)                    # (T, S, 29)
        sp3 = bank.feat[2]
        reg = np.zeros((self.T, self.S), dtype=np.int64)
        for si in range(self.S):
            cols = {n: self.raw[2][si][:, j] for j, n in enumerate(bank.columns[2])}
            reg[:, si] = sp3.classify_regime(cols)
        self.regime = reg                                                                            # past-only stock regime code (-1 unknown)
        bf = sp3.build_features(self.bench, self.bench)
        self.bench_regime = sp3.classify_regime({n: bf[n].to_numpy(float) for n in sp3.FEATURE_COLUMNS})   # past-only benchmark regime
        eye = np.vstack([np.eye(len(REGIME_NAMES)), np.zeros((1, len(REGIME_NAMES)))])
        self.regime_oh = eye[np.where(reg >= 0, reg, len(REGIME_NAMES))].astype(np.float32)          # (T, S, 5)
        self.gate = {n: self.get(n).T for n in REQUIRED_RISK_FEATURES}                               # (T, S) each
        self.mean = self.std = self.mkt = None

    def get(self, name: str) -> np.ndarray:
        """Raw feature array (S, T) by name."""
        p, j = self.name2loc[name]
        return self.raw[p][:, :, j]

    def fit_normalizer(self, k_end: int):
        """z-score statistics from rows [valid_start, k_end] only (training data), then normalise everything."""
        x = self.mkt_raw[self.valid_start:k_end + 1].reshape(-1, len(MARKET_FEATURES))
        x = np.where(np.isfinite(x), x, np.nan)
        self.mean = np.nanmean(x, axis=0); self.std = np.nanstd(x, axis=0)
        self.std = np.where(self.std < 1e-8, 1.0, self.std)
        z = np.clip((self.mkt_raw - self.mean) / self.std, -5.0, 5.0)
        self.mkt = np.nan_to_num(z, nan=0.0).astype(np.float32)

    def segments(self, policy_train_end, policy_val_end, meta_val_start, mode: str = "policy_oos") -> dict:
        """Chronological meta-train / meta-val / meta-test index ranges (k0, k1): decisions are taken for k0 <= k < k1."""
        ix = lambda d: int(self.cal.searchsorted(pd.Timestamp(d), side="right")) - 1       # last index with date <= d
        te, ve = ix(policy_train_end), ix(policy_val_end)
        if mode == "policy_oos":          # selector never sees data the five policies were trained on
            tr0, tr1 = te + 1, ix(meta_val_start)
            va0, va1 = tr1 + 1, ve
        elif mode == "full_history":      # includes the policies' own training data (in-sample proposals!)
            tr0, tr1 = self.valid_start, ix(policy_train_end)
            va0, va1 = tr1 + 1, ve
        else:
            raise ValueError("mode must be 'policy_oos' or 'full_history'")
        te0, te1 = ve + 1, self.T - 1
        segs = {"train": (max(tr0, self.valid_start), tr1), "val": (va0, va1), "test": (te0, te1)}
        for n, (a, b) in segs.items():
            if b - a < 40:
                raise ValueError(f"segment '{n}' has only {b - a} decisions; widen the date range or change meta_val_start")
        return segs


# =====================================================================================================
# Investors (synthetic by default)
# =====================================================================================================
BEHAVIOR_KEYS = ["freq_hold", "freq_buy", "freq_sell", "freq_exit", "avg_holding_norm", "position_size", "trade_freq_norm",
                 "vol_entry_share", "dd_exit_share"]


@dataclass
class Investor:
    """Declared risk profile and historical behaviour statistics of one investor.

    ``synthetic=True`` means every number was generated by a documented rule from (risk, patience, aggressiveness):
    the selector can then NOT be said to have learned any real individual's behaviour."""
    name: str
    risk_tolerance: float
    patience: float
    aggressiveness: float
    horizon_bars: int
    drawdown_limit: float
    max_position_fraction: float
    allow_short: bool
    behavior: dict
    authorised: bool = True
    synthetic: bool = True

    @property
    def profile(self) -> np.ndarray:
        return np.array([self.risk_tolerance, self.patience, self.aggressiveness], dtype=np.float32)

    @property
    def entry_fraction(self) -> float:
        return float(self.behavior["position_size"])


def synthetic_behavior(risk: float, pat: float, agg: float, allow_short: bool, rng=None) -> dict:
    """Documented rule that turns a risk profile into 'historical' behaviour statistics (all in [0, 1])."""
    j = (lambda: 1.0 + 0.1 * rng.standard_normal()) if rng is not None else (lambda: 1.0)
    trade_freq = np.clip((0.3 + 2.2 * agg * (1.0 - 0.6 * pat)) * j(), 0.1, 4.0)          # entries per 20 bars
    entries_per_bar = trade_freq / 20.0
    short_share = (0.2 * agg) if allow_short else 0.0
    b = {"freq_buy": entries_per_bar * (1.0 - short_share), "freq_sell": entries_per_bar * short_share, "freq_exit": entries_per_bar,
         "avg_holding_norm": np.clip((8.0 + 50.0 * pat) * j() / 60.0, 0.05, 1.0),
         "position_size": float(np.clip((0.15 + 0.5 * risk) * j(), 0.1, 0.6)),
         "trade_freq_norm": trade_freq / 4.0,
         "vol_entry_share": float(np.clip((0.05 + 0.5 * risk) * j(), 0.0, 1.0)),
         "dd_exit_share": float(np.clip((0.9 - 0.6 * risk) * j(), 0.0, 1.0))}
    b["freq_hold"] = float(max(0.0, 1.0 - b["freq_buy"] - b["freq_sell"] - b["freq_exit"]))
    return {k: float(b[k]) for k in BEHAVIOR_KEYS}


def make_investor(name: str, risk: float, pat: float, agg: float, rng=None, behavior: dict | None = None) -> Investor:
    risk, pat, agg = (float(np.clip(x, 0.0, 1.0)) for x in (risk, pat, agg))
    allow_short = bool(risk > 0.65 and agg > 0.6)
    return Investor(name=name, risk_tolerance=risk, patience=pat, aggressiveness=agg, horizon_bars=int(round(20 + 100 * pat)),
                    drawdown_limit=0.06 + 0.16 * risk, max_position_fraction=0.25 + 0.35 * risk, allow_short=allow_short,
                    behavior=behavior or synthetic_behavior(risk, pat, agg, allow_short, rng))


ARCHETYPES = {"conservative": (0.20, 0.80, 0.20), "balanced": (0.50, 0.50, 0.50), "aggressive": (0.85, 0.25, 0.85)}


def archetype_investors() -> dict:
    return {n: make_investor(n, *v) for n, v in ARCHETYPES.items()}


def sample_investor(rng: np.random.Generator) -> Investor:
    """Training population: archetype + noise (SYNTHETIC)."""
    name = rng.choice(list(ARCHETYPES))
    r, p, a = (np.clip(np.array(ARCHETYPES[name]) + 0.12 * rng.standard_normal(3), 0.0, 1.0))
    return make_investor(f"synthetic_{name}", r, p, a, rng)


def behavior_stats_from_trades(trades: pd.DataFrame, bars_per_period: int | None = None) -> dict:
    """Behaviour statistics from a REAL trade history (columns: action in HOLD/BUY/SELL/EXIT per decision bar, plus optional
    holding_bars, position_size, entered_in_high_vol, exited_in_drawdown). Use this to replace the synthetic behaviour."""
    a = trades["action"].astype(str).str.upper()
    n = max(len(a), 1)
    freq = {f"freq_{k.lower()}": float((a == k).sum() / n) for k in ACTION_NAMES}
    entries = freq["freq_buy"] + freq["freq_sell"]
    out = {**freq, "avg_holding_norm": float(np.clip(trades.get("holding_bars", pd.Series([20])).mean() / 60.0, 0.0, 1.0)),
           "position_size": float(np.clip(trades.get("position_size", pd.Series([0.3])).mean(), 0.0, 1.0)),
           "trade_freq_norm": float(np.clip(entries * 20.0 / 4.0, 0.0, 1.0)),
           "vol_entry_share": float(trades.get("entered_in_high_vol", pd.Series([0.2])).mean()),
           "dd_exit_share": float(trades.get("exited_in_drawdown", pd.Series([0.3])).mean())}
    return {k: float(out[k]) for k in BEHAVIOR_KEYS}


# =====================================================================================================
# Account (vectorised sleeves) and behaviour tracking
# =====================================================================================================
@dataclass
class ExecConfig:
    initial_capital: float = 100000.0
    fee_bps: float = 5.0
    slippage_bps: float = 5.0
    short_borrow_bps_annual: float = 50.0
    min_trade_fraction: float = 0.01
    dd_lookback: int = 120            # drawdown is measured below the TRAILING high of this many bars (a halt expires after a cooling-off period)


class AccountBatch:
    """B independent single-asset accounts. Orders fill at the NEXT bar's open; marking happens at the close.
    Drawdown = 1 - equity / trailing high (last `dd_lookback` bars). An all-time peak would lock an account that went to cash
    after a loss out of the market forever, because cash cannot recover to the old peak."""

    def __init__(self, B: int, cfg: ExecConfig):
        self.cfg, self.B = cfg, B
        self.fee, self.slip = cfg.fee_bps / 1e4, cfg.slippage_bps / 1e4
        self.borrow = cfg.short_borrow_bps_annual / 1e4 / 252.0
        self.cash = np.full(B, cfg.initial_capital); self.shares = np.zeros(B); self.entry_px = np.zeros(B)
        self.hold = np.zeros(B, dtype=int); self.eq = np.full(B, cfg.initial_capital); self.peak = self.eq.copy()
        self.hist = [self.eq.copy()]; self.last_ret = np.zeros(B)
        self.lookback = int(cfg.dd_lookback)

    def execute(self, action: np.ndarray, o: np.ndarray, frac: np.ndarray) -> dict:
        B, F, S = self.B, self.fee, self.slip
        traded = np.zeros(B); cost = np.zeros(B); trade_ret = np.full(B, np.nan); hold_before = self.hold.copy()
        sh = self.shares
        ex = (action == EXIT) & (sh != 0)
        px = np.where(sh > 0, o * (1 - S), o * (1 + S))
        fee_amt = F * np.abs(sh) * px
        self.cash = np.where(ex, self.cash + sh * px - fee_amt, self.cash)
        cost = np.where(ex, fee_amt + S * np.abs(sh) * o, cost)
        traded = np.where(ex, np.abs(sh) * px, traded)
        trade_ret = np.where(ex, np.sign(sh) * (px / np.where(self.entry_px > 0, self.entry_px, 1.0) - 1.0), trade_ret)
        self.shares = np.where(ex, 0.0, sh); self.entry_px = np.where(ex, 0.0, self.entry_px)
        eq_open = self.cash + self.shares * o
        flat = self.shares == 0
        buy, sell = (action == BUY) & flat, (action == SELL) & flat
        target = frac * eq_open
        n_b = np.minimum(target, self.cash / (1 + F))
        ok_b = buy & (n_b >= self.cfg.min_trade_fraction * eq_open)
        ok_s = sell & (target >= self.cfg.min_trade_fraction * eq_open)
        pxb, pxs = o * (1 + S), o * (1 - S)
        self.shares = np.where(ok_b, n_b / pxb, np.where(ok_s, -target / pxs, self.shares))
        self.cash = np.where(ok_b, self.cash - n_b * (1 + F), np.where(ok_s, self.cash + target * (1 - F), self.cash))
        self.entry_px = np.where(ok_b, pxb, np.where(ok_s, pxs, self.entry_px))
        notional = np.where(ok_b, n_b, np.where(ok_s, target, 0.0))
        cost = cost + np.where(ok_b | ok_s, F * notional + S * (notional / np.where(ok_b, pxb, pxs)) * o, 0.0)
        traded = traded + notional
        entered = ok_b | ok_s
        return {"traded": traded, "cost": cost, "entered": entered, "exited": ex, "trade_ret": trade_ret,
                "hold_before_exit": hold_before, "skipped_entry": (buy | sell) & ~entered, "equity_open": eq_open}

    def mark(self, c1: np.ndarray, entered: np.ndarray) -> np.ndarray:
        short = self.shares < 0
        self.cash = self.cash - np.where(short, -self.shares * c1, 0.0) * self.borrow
        new_eq = np.maximum(self.cash + self.shares * c1, 1e-9)
        ret = new_eq / self.eq - 1.0
        self.hold = np.where(entered, 1, np.where(self.shares != 0, self.hold + 1, 0))
        self.eq = new_eq; self.hist.append(new_eq.copy()); self.last_ret = ret
        self.peak = np.stack(self.hist[-(self.lookback + 1):], axis=0).max(axis=0)          # trailing high-water mark, not the all-time peak
        return ret


class BehaviorTracker:
    """Past-only running statistics of the behaviour a method actually produces, compared with the investor's history.
    Consistency is a separate measurable quantity; it says nothing about profitability."""

    def __init__(self, targets: np.ndarray):
        self.target = targets.astype(float)                       # (B, 9) in BEHAVIOR_KEYS order
        self.real = targets.astype(float).copy()                  # start at the investor's history (no initial penalty)
        self.rate = self.target[:, 6] / 5.0                        # running entries-per-bar estimate

    def update(self, final, entered, exited, hold_before, size, vol_pct, dd):
        a = 1.0 / 40.0
        for i, act in enumerate((HOLD, BUY, SELL, EXIT)):
            self.real[:, i] += a * ((final == act).astype(float) - self.real[:, i])
        self.real[:, 4] = np.where(exited, self.real[:, 4] + 0.3 * (np.clip(hold_before / 60.0, 0, 1) - self.real[:, 4]), self.real[:, 4])
        inpos = size > 0
        self.real[:, 5] = np.where(inpos, self.real[:, 5] + (1 / 20.0) * (size - self.real[:, 5]), self.real[:, 5])
        self.rate += (1 / 20.0) * (entered.astype(float) - self.rate)
        self.real[:, 6] = np.clip(5.0 * self.rate, 0.0, 1.0)                   # entries per 20 bars / 4
        volhigh = (np.nan_to_num(vol_pct, nan=0.0) > 0.7).astype(float)
        self.real[:, 7] = np.where(entered, self.real[:, 7] + 0.2 * (volhigh - self.real[:, 7]), self.real[:, 7])
        self.real[:, 8] = np.where(exited, self.real[:, 8] + 0.2 * ((dd > 0.02).astype(float) - self.real[:, 8]), self.real[:, 8])

    def distance(self) -> np.ndarray:
        return np.clip(np.abs(self.real - self.target), 0.0, 1.0).mean(axis=1)

    def components(self) -> np.ndarray:
        return np.abs(self.real - self.target)


# =====================================================================================================
# Reward
# =====================================================================================================
@dataclass
class RewardConfig:
    """All terms are expressed in units of ``sigma_ref`` (a reference daily volatility), so weights are comparable.

    reward = w_perf   * (return - rf) / sigma_ref                                  risk-adjusted performance (realised, net of costs)
           - w_dd     * increase of max(0, drawdown - tolerance) / sigma_ref       drawdown beyond the investor's tolerance
           - w_cost   * (fees + slippage paid) / equity / sigma_ref                explicit extra penalty on transaction costs
           - w_turn   * traded_notional / equity * turnover_charge / sigma_ref     unnecessary turnover
           - w_behav  * behaviour_distance                                         consistency with the investor's HISTORY (not imitation of actions)
           - rejection_penalty  * [selected policy's entry was rejected by the gate]
           - forced_exit_penalty * [gate had to force an exit]
    Only quantities known at the end of the step are used; no hindsight, no oracle policy, no future prices as inputs."""
    w_perf: float = 1.0
    w_drawdown: float = 1.0
    w_cost: float = 0.5
    w_turnover: float = 0.5
    w_behavior: float = 0.1
    sigma_ref: float = 0.01
    dd_tolerance_fraction: float = 0.5
    turnover_charge: float = 0.001
    rejection_penalty: float = 0.2
    forced_exit_penalty: float = 0.0
    rf_annual: float = 0.0
    clip: float = 10.0

    def to_dict(self):
        return asdict(self)


def compute_reward(cfg: RewardConfig, ret, dd_prev, dd_now, dd_limit, cost_frac, traded_frac, beh_dist, blocked, forced) -> dict:
    s = cfg.sigma_ref
    tol = cfg.dd_tolerance_fraction * dd_limit
    comp = {"performance": cfg.w_perf * (ret - cfg.rf_annual / 252.0) / s,
            "drawdown": -cfg.w_drawdown * (np.maximum(0, dd_now - tol) - np.maximum(0, dd_prev - tol)) / s,
            "cost": -cfg.w_cost * cost_frac / s,
            "turnover": -cfg.w_turnover * traded_frac * cfg.turnover_charge / s,
            "behavior": -cfg.w_behavior * beh_dist,
            "rejection": -cfg.rejection_penalty * blocked.astype(float) - cfg.forced_exit_penalty * forced.astype(float)}
    total = np.clip(sum(comp.values()), -cfg.clip, cfg.clip)
    return {"total": total, **comp}


# =====================================================================================================
# Observation schema
# =====================================================================================================
SHADOW_METRICS = ["ret_20", "ret_60", "maxdd_60", "sharpe_20", "position"]
INVESTOR_FEATURES = ["risk_tolerance", "patience", "aggressiveness", "horizon_norm", "drawdown_limit_norm", "max_position_fraction", "allow_short"]
POSITION_FEATURES = ["position", "exposure", "unrealized_pnl_scaled", "holding_scaled", "checkpoint_distance", "drawdown_scaled",
                     "recent_loss_scaled", "equity_growth_scaled"]


def observation_names() -> list:
    names = [f"A.market.{n}" for n in MARKET_FEATURES] + [f"A.regime.{r}" for r in REGIME_NAMES]
    names += [f"B.behavior.{k}" for k in BEHAVIOR_KEYS]
    names += [f"C.investor.{n}" for n in INVESTOR_FEATURES] + [f"C.position.{n}" for n in POSITION_FEATURES]
    for k in POLICY_KEYS:
        names += [f"D1.{k}.{x}" for x in ("p_hold", "p_buy", "p_sell", "p_exit", "confidence")]
    for k in POLICY_KEYS:
        names += [f"D2.{k}.{m}" for m in SHADOW_METRICS]
    return names


OBS_NAMES = observation_names()
OBS_DIM = len(OBS_NAMES)


def obs_mask(kind: str) -> np.ndarray:
    """1 = visible to the selector. 'full' | 'no_profile' | 'no_policy_perf' (ablations)."""
    m = np.ones(OBS_DIM, dtype=np.float32)
    for i, n in enumerate(OBS_NAMES):
        if kind == "no_profile" and (n.startswith("B.") or n.startswith("C.investor.")):
            m[i] = 0.0
        if kind == "no_policy_perf" and n.startswith("D2."):
            m[i] = 0.0
    return m


# =====================================================================================================
# One simulation step over B sleeves
# =====================================================================================================
class SleeveSim:
    def __init__(self, md: MarketData, bank: PolicyBank, gate: MetaRiskController, cfg: ExecConfig, sym_idx, investors: list, k0: int):
        self.md, self.bank, self.gate, self.cfg = md, bank, gate, cfg
        self.sym = np.asarray(sym_idx, dtype=int); self.B = len(self.sym); self.inv = investors
        self.prof = np.stack([i.profile for i in investors]).astype(np.float32)
        self.dd_limit = np.array([i.drawdown_limit for i in investors]); self.max_frac = np.array([i.max_position_fraction for i in investors])
        self.allow_short = np.array([i.allow_short for i in investors]); self.authorised = np.array([i.authorised for i in investors])
        self.horizon = np.array([i.horizon_bars for i in investors], dtype=float)
        self.entry_frac = gate.entry_fraction(np.array([i.entry_fraction for i in investors]), self.max_frac)
        self.acct = AccountBatch(self.B, cfg)
        self.beh = BehaviorTracker(np.stack([[i.behavior[k] for k in BEHAVIOR_KEYS] for i in investors]))
        self.k0 = np.broadcast_to(np.asarray(k0), (self.B,)).astype(int).copy()          # per-sleeve start bar
        self.k = self.k0.copy()                                                         # per-sleeve current decision bar
        self.probs = None

    # ---- state at the close of bar k ------------------------------------------------------------
    def context(self) -> dict:
        a = self.acct; c = self.md.close[self.k, self.sym]
        pos = np.sign(a.shares).astype(int); size = np.abs(a.shares * c) / a.eq
        pnl = np.where(pos != 0, pos * (c / np.where(a.entry_px > 0, a.entry_px, 1.0) - 1.0), 0.0)
        pdd = np.maximum(0.0, 1.0 - a.eq / a.peak)
        rloss = np.maximum(0.0, 1.0 - a.eq / a.hist[max(0, len(a.hist) - 21)])
        return {"pos": pos, "size": size, "gross": size, "pnl": pnl, "hold": a.hold.copy(), "pdd": pdd, "rloss": rloss,
                "dret": a.last_ret.copy(), "eq": a.eq.copy()}

    def raw(self, name: str) -> np.ndarray:
        return self.md.get(name)[self.sym, self.k]

    def compute_candidates(self, which=range(N_POLICIES)) -> np.ndarray:
        ctx = self.context()
        self.probs = self.bank.proposals(self.md, self.k, self.sym, ctx, self.prof, which)
        return self.probs

    def proposal_of(self, selected: np.ndarray):
        """Action proposed by the selected policy of each sleeve and its confidence (calibrated max probability)."""
        pr = self.probs[np.arange(self.B), selected]
        act = pr.argmax(axis=1)
        return act, pr[np.arange(self.B), act]

    # ---- observation ---------------------------------------------------------------------------------
    def observe(self, shadow_k: np.ndarray | None) -> np.ndarray:
        """Observation vector (B, OBS_DIM) using only information available at the close of bar k."""
        ctx = self.context(); md = self.md
        A = np.concatenate([md.mkt[self.k, self.sym], md.regime_oh[self.k, self.sym]], axis=1)
        Bb = self.beh.target.astype(np.float32)
        inv = np.stack([self.prof[:, 0], self.prof[:, 1], self.prof[:, 2], self.horizon / 120.0, self.dd_limit / 0.25,
                        self.max_frac, self.allow_short.astype(float)], axis=1)
        t_ep = self.k - self.k0
        ckpt = (self.horizon - (t_ep % self.horizon)) / self.horizon
        pos_f = np.stack([ctx["pos"], ctx["size"], np.clip(ctx["pnl"] / 0.1, -3, 3), np.minimum(ctx["hold"], 60) / 20.0, ckpt,
                          np.clip(ctx["pdd"] / 0.1, 0, 3), np.clip(ctx["rloss"] / 0.05, 0, 3),
                          np.clip((ctx["eq"] / self.cfg.initial_capital - 1.0) / 0.2, -3, 3)], axis=1)
        if self.probs is None:
            self.compute_candidates()
        D1 = np.concatenate([self.probs, self.probs.max(axis=2, keepdims=True)], axis=2).reshape(self.B, -1)   # (B, 25): per policy 4 probs + confidence
        D2 = np.zeros((self.B, N_POLICIES * len(SHADOW_METRICS)), dtype=np.float32) if shadow_k is None else shadow_k.reshape(self.B, -1)
        obs = np.concatenate([A, Bb, inv, pos_f, D1, D2], axis=1).astype(np.float32)
        return np.clip(np.nan_to_num(obs, nan=0.0), -10.0, 10.0)

    # ---- one decision: proposal -> gate -> execution ---------------------------------------------------
    def apply(self, proposed: np.ndarray, gate_on: bool = True) -> dict:
        k, ctx = self.k, self.context()
        proposed = np.asarray(proposed).astype(int)
        feats = {n: self.md.gate[n][k, self.sym] for n in REQUIRED_RISK_FEATURES}
        if gate_on:
            final, reasons = self.gate.review_batch(
                proposed, ctx["pos"], ctx["pnl"], ctx["hold"], feats,
                {"drawdown": ctx["pdd"], "gross_exposure": ctx["gross"], "daily_return": ctx["dret"], "recent_loss": ctx["rloss"],
                 "entry_fraction": self.entry_frac},
                {"drawdown_limit": self.dd_limit, "max_position_fraction": self.max_frac, "allow_short": self.allow_short,
                 "authorised": self.authorised})
        else:
            valid = valid_action_mask(ctx["pos"], True)
            final = np.where(valid[np.arange(self.B), proposed], proposed, HOLD)
            reasons = np.zeros((self.B, len(REASON_NAMES)), dtype=bool)
        blocked = ((proposed == BUY) | (proposed == SELL)) & (final == HOLD)            # the selected policy's entry was rejected
        forced = (final == EXIT) & (proposed != EXIT)                                    # the gate forced a de-risking exit
        eq_prev, dd_prev = ctx["eq"], ctx["pdd"]
        o, c1 = self.md.open[k + 1, self.sym], self.md.close[k + 1, self.sym]
        ex = self.acct.execute(final, o, self.entry_frac)
        ret = self.acct.mark(c1, ex["entered"])
        dd_now = np.maximum(0.0, 1.0 - self.acct.eq / self.acct.peak)
        size_now = np.abs(self.acct.shares * c1) / self.acct.eq
        vol_pct = np.maximum(feats["atr_pct_pctile_252"], feats["rv20_pctile_252"])
        self.beh.update(final, ex["entered"], ex["exited"], ex["hold_before_exit"], size_now, vol_pct, dd_prev)
        self.k = k + 1
        self.probs = None
        return {"proposed": proposed, "final": final, "reasons": reasons, "blocked": blocked, "forced": forced, "ret": ret, "equity": self.acct.eq.copy(),
                "dd_prev": dd_prev, "dd_now": dd_now, "cost_frac": ex["cost"] / eq_prev, "traded_frac": ex["traded"] / eq_prev, "entered": ex["entered"],
                "exited": ex["exited"], "trade_ret": ex["trade_ret"], "beh_dist": self.beh.distance(), "skipped_entry": ex["skipped_entry"]}

    def reward(self, rc: RewardConfig, r: dict) -> dict:
        return compute_reward(rc, r["ret"], r["dd_prev"], r["dd_now"], self.dd_limit, r["cost_frac"], r["traded_frac"], r["beh_dist"], r["blocked"], r["forced"])

    # ---- export for the public inference interface -------------------------------------------------------
    def export_observation(self, b: int, shadow_k: np.ndarray | None, mask: np.ndarray | None = None) -> dict:
        """Everything MetaAgent needs for one sleeve: the selector vector, the five policy states (public predict() format),
        and the risk context. Contains only information available at the close of the current bar."""
        ctx = self.context(); k, s = int(self.k[b]), self.sym[b]
        vec = self.observe(shadow_k)[b]
        L = self.bank.seq_len
        states = []
        for p in range(N_POLICIES):
            st = {"sequence": self.md.raw[p][s, k - L + 1:k + 1].copy(), "position": int(ctx["pos"][b]), "profile": self.prof[b].tolist()}
            if p >= 1:
                st.update(unrealized_pnl=float(ctx["pnl"][b]), holding_bars=int(ctx["hold"][b]))
            if p == 4:
                st.update(portfolio_drawdown=float(ctx["pdd"][b]), gross_exposure=float(ctx["gross"][b]), position_size=float(ctx["size"][b]),
                          recent_loss=float(ctx["rloss"][b]))
            states.append(st)
        return {"vector": vec, "policy_inputs": states,
                "risk_context": {"features": {n: float(self.md.gate[n][k, s]) for n in REQUIRED_RISK_FEATURES},
                                 "position": int(ctx["pos"][b]), "unrealized_pnl": float(ctx["pnl"][b]), "holding_bars": int(ctx["hold"][b]),
                                 "account": {"drawdown": float(ctx["pdd"][b]), "gross_exposure": float(ctx["gross"][b]), "daily_return": float(ctx["dret"][b]),
                                             "recent_loss": float(ctx["rloss"][b]), "entry_fraction": float(self.entry_frac[b])},
                                 "investor": {"drawdown_limit": float(self.dd_limit[b]), "max_position_fraction": float(self.max_frac[b]),
                                              "allow_short": bool(self.allow_short[b]), "authorised": bool(self.authorised[b])}},
                "meta": {"symbol": self.md.symbols[s], "date": str(self.md.cal[k].date()), "investor": self.inv[b].name}}


# =====================================================================================================
# Past-only simulated performance of every policy (candidate-policy information)
# =====================================================================================================
def shadow_metrics_from_history(E: np.ndarray, pos: np.ndarray) -> np.ndarray:
    """E: (t+1, S) shadow equity up to the current close; pos: (S,) shadow position -> (S, 5) metrics. Past-only."""
    t = E.shape[0] - 1
    ret = lambda n: E[t] / E[max(0, t - n)] - 1.0
    win = E[max(0, t - 60):t + 1]
    dd60 = (1.0 - win / np.maximum.accumulate(win, axis=0)).max(axis=0)
    r = np.diff(E[max(0, t - 20):t + 1], axis=0) / E[max(0, t - 20):t, :] if t >= 2 else np.zeros((1, E.shape[1]))
    sd = r.std(axis=0)
    sharpe = np.where(sd > 1e-9, r.mean(axis=0) / np.where(sd > 1e-9, sd, 1.0) * np.sqrt(20.0), 0.0)
    return np.stack([np.clip(ret(20) / 0.1, -3, 3), np.clip(ret(60) / 0.2, -3, 3), np.clip(dd60 / 0.1, 0, 3), np.clip(sharpe, -3, 3) / 3.0, pos.astype(float)], axis=1)


def run_shadow(md: MarketData, bank: PolicyBank, gate: MetaRiskController, cfg: ExecConfig, seg: tuple, ref: Investor):
    """Simulate each policy ALONE on every symbol from the start of the segment with a reference investor (same costs, same gate).
    Returns shadow features (K+1, S, 5 policies, 5 metrics) where row t refers to the close of bar k0+t, using only the past."""
    k0, k1 = seg; S = md.S
    sims = [SleeveSim(md, bank, gate, cfg, np.arange(S), [ref] * S, k0) for _ in range(N_POLICIES)]
    out = np.zeros((k1 - k0 + 1, S, N_POLICIES, len(SHADOW_METRICS)), dtype=np.float32)
    for t, k in enumerate(range(k0, k1 + 1)):
        for p, sim in enumerate(sims):
            E = np.stack(sim.acct.hist, axis=0)
            out[t, :, p] = shadow_metrics_from_history(E, np.sign(sim.acct.shares))
        if k == k1:
            break
        for p, sim in enumerate(sims):
            sim.compute_candidates(which=[p])
            act, _ = sim.proposal_of(np.full(S, p))
            sim.apply(act)
    return out


# =====================================================================================================
# Gymnasium environment
# =====================================================================================================
class MetaTradingEnv(gym.Env):
    """One sleeve per episode. Action = which of the five frozen policies controls the next decision.

    step(): proposal of the selected policy -> deterministic risk gate -> execution at the next open -> reward -> advance one bar."""
    metadata = {"render_modes": []}

    def __init__(self, md, bank, gate, exec_cfg, reward_cfg, shadow: np.ndarray, seg: tuple, mode: str = "train",
                 episode_length: int = 120, observation_mask: np.ndarray | None = None, eval_investors: dict | None = None, seed: int = 0):
        super().__init__()
        self.md, self.bank, self.gate, self.ecfg, self.rcfg = md, bank, gate, exec_cfg, reward_cfg
        self.shadow, self.seg, self.mode, self.ep_len = shadow, seg, mode, episode_length
        self.mask = np.ones(OBS_DIM, dtype=np.float32) if observation_mask is None else observation_mask.astype(np.float32)
        self.eval_investors = eval_investors or archetype_investors()
        self.combos = [(s, n) for s in range(md.S) for n in self.eval_investors]
        self._combo_i = 0
        self.observation_space = spaces.Box(-10.0, 10.0, shape=(OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.Discrete(N_POLICIES)
        self.rng = np.random.default_rng(seed)
        self.sim = None; self.end = None

    def _obs(self):
        sh = self.shadow[self.sim.k - self.seg[0], self.sim.sym]                       # (1, 5, 5)
        return (self.sim.observe(sh)[0] * self.mask).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        k0, k1 = self.seg
        opt = options or {}
        if "symbol" in opt:                                                             # explicit episode (tests / reproducibility)
            s = self.md.symbols.index(opt["symbol"]); inv = opt["investor"]; start = int(opt["start"]); end = int(opt["end"])
        elif self.mode == "train":
            s = int(self.rng.integers(self.md.S)); inv = sample_investor(self.rng)
            L = min(self.ep_len, k1 - k0); start = int(self.rng.integers(k0, k1 - L + 1)); end = start + L
        else:
            s, name = self.combos[self._combo_i % len(self.combos)]; self._combo_i += 1
            inv = self.eval_investors[name]; start, end = k0, k1
        self.sim = SleeveSim(self.md, self.bank, self.gate, self.ecfg, [s], [inv], start)
        self.end = end
        self.sim.compute_candidates()
        return self._obs(), {"symbol": self.md.symbols[s], "investor": inv.name}

    def step(self, action):
        sel = np.array([int(action)])
        proposed, conf = self.sim.proposal_of(sel)
        r = self.sim.apply(proposed)
        rw = self.sim.reward(self.rcfg, r)
        truncated = bool(self.sim.k[0] >= self.end)
        if not truncated:
            self.sim.compute_candidates()
        obs = self._obs() if not truncated else np.zeros(OBS_DIM, dtype=np.float32)
        info = {"selected": int(action), "proposed": int(r["proposed"][0]), "final": int(r["final"][0]), "blocked": bool(r["blocked"][0]),
                "forced": bool(r["forced"][0]), "reasons": [REASON_NAMES[j] for j in np.where(r["reasons"][0])[0]], "equity": float(r["equity"][0]),
                "ret": float(r["ret"][0]), "reward_components": {k: float(v[0]) for k, v in rw.items()}}
        return obs, float(rw["total"][0]), False, bool(truncated), info


# =====================================================================================================
# Batched vectorised environment for Stable-Baselines3 (n sleeves stepped together; each has its own symbol, investor, start bar)
# =====================================================================================================
from stable_baselines3.common.vec_env.base_vec_env import VecEnv  # noqa: E402


class BatchedMetaVecEnv(VecEnv):
    """n copies of MetaTradingEnv stepped in ONE batch (same logic, same SleeveSim), so the five policies run once per step for
    all sleeves. Episodes have equal length and finish together; finished sleeves are re-drawn (new symbol, investor, start bar)."""

    def __init__(self, md, bank, gate, exec_cfg, reward_cfg, shadow, seg, n_envs, mode="train", episode_length=120,
                 observation_mask=None, eval_investors=None, seed=0):
        self.md, self.bank, self.gate, self.ecfg, self.rcfg = md, bank, gate, exec_cfg, reward_cfg
        self.shadow, self.seg, self.mode, self.ep_len = shadow, seg, mode, episode_length
        self.mask = np.ones(OBS_DIM, dtype=np.float32) if observation_mask is None else observation_mask.astype(np.float32)
        self.eval_investors = eval_investors or archetype_investors()
        self.rng = np.random.default_rng(seed)
        self._combo_i = 0
        super().__init__(n_envs, spaces.Box(-10.0, 10.0, shape=(OBS_DIM,), dtype=np.float32), spaces.Discrete(N_POLICIES))
        self.sim = None; self._actions = None; self._ep_r = np.zeros(n_envs); self._ep_l = 0

    def _specs(self):
        k0, k1 = self.seg; n = self.num_envs
        if self.mode == "train":
            sym = self.rng.integers(self.md.S, size=n); invs = [sample_investor(self.rng) for _ in range(n)]
            L = min(self.ep_len, k1 - k0); start = self.rng.integers(k0, k1 - L + 1, size=n); end = start + L
        else:                                                                          # deterministic sweep over (symbol, archetype)
            names = list(self.eval_investors); combos = [(s, nm) for s in range(self.md.S) for nm in names]
            sel = [combos[(self._combo_i + i) % len(combos)] for i in range(n)]; self._combo_i += n
            sym = np.array([c[0] for c in sel]); invs = [self.eval_investors[c[1]] for c in sel]
            start = np.full(n, k0); end = np.full(n, k1)
        return sym, invs, start, end

    def _obs(self):
        return (self.sim.observe(self.shadow[self.sim.k - self.seg[0], self.sim.sym]) * self.mask).astype(np.float32)

    def reset(self):
        sym, invs, start, self.end = self._specs()
        self.sim = SleeveSim(self.md, self.bank, self.gate, self.ecfg, sym, invs, start)
        self.sim.compute_candidates()
        self._ep_r[:] = 0.0; self._ep_l = 0
        self.reset_infos = [{} for _ in range(self.num_envs)]
        return self._obs()

    def step_async(self, actions):
        self._actions = np.asarray(actions).astype(int)

    def step_wait(self):
        sim = self.sim
        proposed, _ = sim.proposal_of(self._actions)
        r = sim.apply(proposed)
        rw = sim.reward(self.rcfg, r)
        self._ep_r += rw["total"]; self._ep_l += 1
        done = bool((sim.k >= self.end).all())
        infos = [{"selected": int(self._actions[i]), "blocked": bool(r["blocked"][i]), "forced": bool(r["forced"][i]), "ret": float(r["ret"][i])}
                 for i in range(self.num_envs)]
        sim.compute_candidates()
        obs = self._obs()
        if done:
            for i in range(self.num_envs):
                infos[i]["terminal_observation"] = obs[i].copy(); infos[i]["TimeLimit.truncated"] = True
                infos[i]["episode"] = {"r": float(self._ep_r[i]), "l": self._ep_l}
            rewards = rw["total"].astype(np.float32); dones = np.ones(self.num_envs, dtype=bool)
            obs = self.reset()
            return obs, rewards, dones, infos
        return obs, rw["total"].astype(np.float32), np.zeros(self.num_envs, dtype=bool), infos

    def close(self): pass
    def get_attr(self, attr_name, indices=None): return [getattr(self, attr_name, None)] * len(self._get_indices(indices))
    def set_attr(self, attr_name, value, indices=None): setattr(self, attr_name, value)
    def env_method(self, method_name, *args, indices=None, **kwargs): return [None] * len(self._get_indices(indices))
    def env_is_wrapped(self, wrapper_class, indices=None): return [False] * len(self._get_indices(indices))
    def seed(self, seed=None):
        self.rng = np.random.default_rng(seed); return [seed] * self.num_envs


# =====================================================================================================
# Batch evaluation of any selector over a chronological segment
# =====================================================================================================
class Selector:
    """kind: 'policy' (returns a policy index per sleeve) or 'direct' (returns actions directly, e.g. buy-and-hold)."""
    def __init__(self, name, fn, kind="policy", needs_all=True, gate_on=True):
        self.name, self.fn, self.kind, self.needs_all, self.gate_on = name, fn, kind, needs_all, gate_on


def run_selector(md, bank, gate, ecfg, rcfg, seg, shadow, selector: Selector, investors: dict, mask=None, record=False):
    """All symbols x all investors in parallel (B = S * A sleeves) over decisions k0..k1-1. Returns per-step arrays."""
    k0, k1 = seg; names = list(investors); A = len(names); S = md.S
    sym = np.repeat(np.arange(S), A); invs = [investors[names[i % A]] for i in range(S * A)]
    sim = SleeveSim(md, bank, gate, ecfg, sym, invs, k0); B = sim.B
    K = k1 - k0
    R = {"equity": np.zeros((K, B)), "ret": np.zeros((K, B)), "reward": np.zeros((K, B)), "selected": np.full((K, B), -1), "proposed": np.zeros((K, B), int),
         "final": np.zeros((K, B), int), "blocked": np.zeros((K, B), bool), "forced": np.zeros((K, B), bool), "beh": np.zeros((K, B)),
         "traded": np.zeros((K, B)), "cost": np.zeros((K, B)), "exposure": np.zeros((K, B)), "entered": np.zeros((K, B), bool),
         "skipped": np.zeros((K, B), bool), "sym": sym, "inv": np.array([names[i % A] for i in range(B)]), "comp": {}, "reasons": []}
    probs_log = np.zeros((K, B, N_POLICIES)) if record else None
    for t, k in enumerate(range(k0, k1)):
        sim.compute_candidates(range(N_POLICIES) if selector.needs_all else [])
        sh = shadow[t][sym] if shadow is not None else None                  # shadow[t]: (S, 5, 5)
        if selector.kind == "policy":
            obs = sim.observe(sh) if selector.needs_all else None
            if selector.needs_all:
                obs = obs * (np.ones(OBS_DIM, dtype=np.float32) if mask is None else mask)
            sel = np.asarray(selector.fn(obs, sim)).astype(int)
            if not selector.needs_all:                                     # only the selected policy is evaluated
                ctx = sim.context()
                sim.probs = np.zeros((B, N_POLICIES, 4), dtype=np.float32)
                for p in np.unique(sel):
                    m = sel == p
                    pr = bank.proposals(md, sim.k[m], sim.sym[m], {kk: v[m] for kk, v in ctx.items()}, sim.prof[m], which=[p])
                    sim.probs[m, p] = pr[:, p]
            proposed, conf = sim.proposal_of(sel)
            if record:
                probs_log[t] = getattr(selector, "last_probs", np.zeros((B, N_POLICIES)))
        else:
            sel = np.full(B, -1); proposed = np.asarray(selector.fn(None, sim)).astype(int)
        r = sim.apply(proposed, gate_on=selector.gate_on)
        rw = sim.reward(rcfg, r)
        for key, v in (("equity", r["equity"] / ecfg.initial_capital), ("ret", r["ret"]), ("reward", rw["total"]), ("beh", r["beh_dist"]),
                       ("traded", r["traded_frac"]), ("cost", r["cost_frac"])):
            R[key][t] = v
        R["selected"][t] = sel; R["proposed"][t] = r["proposed"]; R["final"][t] = r["final"]; R["blocked"][t] = r["blocked"]; R["forced"][t] = r["forced"]
        R["entered"][t] = r["entered"]; R["skipped"][t] = r["skipped_entry"]
        R["exposure"][t] = np.abs(sim.acct.shares * md.close[sim.k, sim.sym]) / sim.acct.eq
        for kname, v in rw.items():
            R["comp"].setdefault(kname, np.zeros((K, B)))[t] = v
        if record:
            R["reasons"].append(r["reasons"].copy())
    R["probs"] = probs_log; R["beh_components"] = sim.beh.components(); R["investors"] = invs
    return R


# =====================================================================================================
# Inference interface
# =====================================================================================================
class MetaAgent:
    """Frozen PPO selector + five frozen policies + deterministic gate. Selection, policy inference, risk validation and
    execution are separate methods; ``predict_action`` chains the first three (it never executes anything)."""

    def __init__(self, model, bank: PolicyBank, gate: MetaRiskController, config: dict, mask: np.ndarray):
        self.model, self.bank, self.gate, self.config, self.mask = model, bank, gate, config, mask
        self.log: list = []

    # 1) model selection -------------------------------------------------------------------------------
    def select_policy(self, observation) -> dict:
        vec = np.asarray(observation["vector"] if isinstance(observation, dict) else observation, dtype=np.float32).reshape(-1)
        if vec.size != OBS_DIM:
            raise ValueError(f"observation must have {OBS_DIM} entries; got {vec.size}")
        vec = vec * self.mask
        action, _ = self.model.predict(vec, deterministic=True)
        obs_t, _ = self.model.policy.obs_to_tensor(vec)
        with torch.no_grad():
            probs = self.model.policy.get_distribution(obs_t).distribution.probs.cpu().numpy()[0]
        idx = int(action)
        return {"policy_id": idx, "policy_name": POLICY_NAMES[idx], "policy_key": POLICY_KEYS[idx],
                "selection_probabilities": {POLICY_NAMES[i]: float(probs[i]) for i in range(N_POLICIES)}}

    # 2) policy inference (the chosen frozen policy proposes) ---------------------------------------------
    def propose(self, policy_id: int, observation: dict) -> dict:
        out = self.bank.policies[policy_id].predict(observation["policy_inputs"][policy_id])        # public predict() of the saved policy
        return {"action": int(out["action"]), "action_name": out["action_name"], "confidence": float(out["confidence"]),
                "action_probs": [float(x) for x in out["action_probs"]]}

    # 3) deterministic risk validation --------------------------------------------------------------------
    def risk_review(self, proposal: dict, observation: dict) -> dict:
        rc = observation["risk_context"]
        return self.gate.review_one(proposal["action"], rc["position"], rc["unrealized_pnl"], rc["holding_bars"], rc["features"], rc["account"], rc["investor"])

    def predict_action(self, observation: dict) -> dict:
        sel = self.select_policy(observation)
        prop = self.propose(sel["policy_id"], observation)
        verdict = self.risk_review(prop, observation)
        rec = {**observation.get("meta", {}), "selected_policy_id": sel["policy_id"], "selected_policy": sel["policy_name"],
               **{f"p_select_{POLICY_KEYS[i]}": sel["selection_probabilities"][POLICY_NAMES[i]] for i in range(N_POLICIES)},
               "proposed_action": prop["action_name"], "proposal_confidence": prop["confidence"], "final_action": verdict["final_action_name"],
               "risk_approved": verdict["approved"], "rejection_reasons": ";".join(verdict["reasons"])}
        self.log.append(rec)
        return {"selected_policy": sel, "proposal": prop, "risk_decision": verdict, "final_action": verdict["final_action"],
                "final_action_name": verdict["final_action_name"], "rejection_reason": verdict["reasons"] or None}

    def save_selection_log(self, path) -> Path:
        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(self.log).to_csv(path, index=False)
        return path


_AGENT: MetaAgent | None = None


def locate_policy_folders(root) -> dict:
    """Find the folder of each policy under `root` (every policy zip has files with identical names, so each needs its own folder)."""
    root = Path(root); found, missing = {}, {}
    for key in POLICY_KEYS:
        pt = POLICY_SPEC[key]["pt"]
        hits = [p.parent for p in root.rglob(pt)]
        if hits:
            found[key] = hits[0]
        else:
            missing[key] = pt
    if missing:
        raise FileNotFoundError(f"missing policy artifacts under {root}: {missing}")
    return found


def load_meta_agent(path, policy_dir=None, device: str = "cpu") -> MetaAgent:
    """Load the PPO selector (zip), its sidecar JSON files and the five frozen policies. `path` = meta_rl_selector.zip or its folder."""
    from stable_baselines3 import PPO
    global _AGENT
    path = Path(path); zpath = path / "meta_rl_selector.zip" if path.is_dir() else path
    if not zpath.exists():
        raise FileNotFoundError(f"selector checkpoint not found: {zpath}")
    cfg = json.loads((zpath.parent / "meta_rl_config.json").read_text())
    folders = locate_policy_folders(policy_dir or cfg["policy_artifact_root"])
    bank = PolicyBank(folders, device="cpu")
    gate = MetaRiskController(MetaRiskLimits.from_dict(json.loads((zpath.parent / "risk_controller_config.json").read_text())["limits"]))
    model = PPO.load(str(zpath), device=device)
    mask = obs_mask(cfg["observation_mask"])
    _AGENT = MetaAgent(model, bank, gate, cfg, mask)
    return _AGENT


def select_policy(observation) -> dict:
    return _AGENT.select_policy(observation)


def predict_action(observation) -> dict:
    return _AGENT.predict_action(observation)


def save_selection_log(path):
    return _AGENT.save_selection_log(path)


In [ ]:
import meta_rl_risk, meta_rl_core
importlib.reload(meta_rl_risk); importlib.reload(meta_rl_core)
import meta_rl_core as M
from meta_rl_risk import (HOLD, BUY, SELL, EXIT, ACTION_NAMES, REASON_NAMES, FORCED_EXIT_REASONS, REQUIRED_RISK_FEATURES,
                          MetaRiskController, MetaRiskLimits, valid_action_mask)
print("modules loaded:", M.__file__, "|", meta_rl_risk.__file__)

## 4. Data download
Daily OHLCV from Yahoo Finance via `yfinance` with **adjusted** prices (`auto_adjust=True`, so Open/High/Low/Close are all split/dividend adjusted). Failures are retried, then reported explicitly: a failed symbol is **skipped, never fabricated**. If the benchmark cannot be downloaded or fewer than three symbols succeed, the notebook stops with a clear error.

Cleaning rules (all counted in the report below): duplicate dates removed, non-positive or missing OHLC rows dropped, rows with `High < Low` dropped, negative volume set to missing. Missing volume is kept as missing (never invented) and handled later by explicit train-median imputation.

In [ ]:
import yfinance as yf

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = df.columns.get_level_values(0)
    return df

def download_symbol(sym: str, start: str, end: str, retries: int = 3, local_dir=None):
    """Return (raw_df, source). Raises RuntimeError with the reason if nothing real can be obtained."""
    last_err = "unknown error"
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(sym, start=start, end=end, interval="1d", auto_adjust=True, progress=False, threads=False)
            df = _flatten_columns(df) if df is not None else None
            if df is not None and len(df) > 0:
                return df, "yfinance"
            last_err = "empty result"
        except Exception as exc:  # noqa: BLE001
            last_err = repr(exc)
        time.sleep(1.5 * attempt)
    if local_dir:
        path = Path(local_dir) / f"{sym}.csv"
        if path.exists():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            return df.loc[start:end], f"local_csv:{path}"
    raise RuntimeError(f"{sym}: download failed after {retries} attempts ({last_err})")

def clean_ohlcv(df: pd.DataFrame, sym: str):
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise ValueError(f"{sym}: missing columns {missing}")
    stats = {"symbol": sym, "raw_rows": int(len(df))}
    df = df[OHLCV].apply(pd.to_numeric, errors="coerce")
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df.index = pd.DatetimeIndex(idx).normalize()
    df.index.name = "Date"
    stats["duplicate_dates_removed"] = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")].sort_index()
    price = ["Open", "High", "Low", "Close"]
    bad = df[price].isna().any(axis=1) | (df[price] <= 0).any(axis=1) | (df["High"] < df["Low"])
    stats["bad_price_rows_dropped"] = int(bad.sum())
    df = df.loc[~bad].copy()
    df.loc[df["Volume"] < 0, "Volume"] = np.nan
    stats["missing_volume_rows"] = int(df["Volume"].isna().sum())
    stats["zero_volume_rows"] = int((df["Volume"] == 0).sum())
    stats["extreme_moves_gt_40pct"] = int((df["Close"].pct_change().abs() > 0.40).sum())
    stats["max_gap_days"] = int(df.index.to_series().diff().dt.days.max()) if len(df) > 1 else 0
    stats["rows"] = int(len(df))
    stats["first_date"] = str(df.index.min().date()) if len(df) else None
    stats["last_date"] = str(df.index.max().date()) if len(df) else None
    return df, stats

download_log, raw_data = [], {}

# benchmark first: it is mandatory
try:
    _b, _src = download_symbol(CONFIG["benchmark"], CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
    bench_df, _st = clean_ohlcv(_b, CONFIG["benchmark"])
    _st.update(source=_src, status="ok"); download_log.append(_st)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(f"Benchmark {CONFIG['benchmark']} could not be obtained, cannot continue: {exc}") from exc

for sym in CONFIG["symbols"]:
    try:
        _raw, _src = download_symbol(sym, CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
        _df, _st = clean_ohlcv(_raw, sym)
        if len(_df) < CONFIG["min_history_rows"]:
            raise ValueError(f"only {len(_df)} clean rows (< min_history_rows={CONFIG['min_history_rows']})")
        _st.update(source=_src, status="ok")
        raw_data[sym] = _df
    except Exception as exc:  # noqa: BLE001
        _st = {"symbol": sym, "status": f"SKIPPED: {exc}"}
        print(f"WARNING: {sym} skipped -> {exc}")
    download_log.append(_st)

if len(raw_data) < 3:
    raise RuntimeError(f"Only {len(raw_data)} symbols downloaded successfully; need at least 3. Check the network / symbols.")

report = pd.DataFrame(download_log).set_index("symbol")
print(f"\nSymbols used ({len(raw_data)}): {list(raw_data)}")
print(f"Symbols skipped: {[s for s in CONFIG['symbols'] if s not in raw_data] or 'none'}")
report

## 5. Load the five frozen policies and validate compatibility
The policies are loaded with their own saved code (`sp0X_model.py` / `sp0X_features.py`) and **frozen** (no gradients, `eval()` mode). Before anything is trained, the notebook verifies, from the saved configs, schemas and preprocessing objects, that the five policies are mutually compatible:

* identical **action mapping** (`0 HOLD, 1 BUY, 2 SELL, 3 EXIT`) and **execution convention** (decide at the close of bar *t*, fill at the open of *t+1*),
* identical **sequence construction** (same `seq_len`) and identical first 30 **sequence feature names and order**,
* consistent **static inputs** (SP-01's 6 columns are a prefix of SP-02..04's 8 columns, which are a prefix of SP-05's 12),
* feature dimensions that match the saved networks and each policy's `preprocessing.pkl`,
* the same training split dates and benchmark across policies (needed for the leakage-free selector splits).

Any failed check stops the notebook.

In [ ]:
bank = M.PolicyBank(POLICY_FOLDERS, device="cpu")                  # frozen policies, loaded from the saved artifacts only
cfgs = {k: json.loads((POLICY_FOLDERS[k] / M.POLICY_SPEC[k]["cfg"]).read_text()) for k in M.POLICY_KEYS}
schemas = {k: json.loads((POLICY_FOLDERS[k] / "feature_schema.json").read_text()) for k in M.POLICY_KEYS}
pres = {k: pickle.load(open(POLICY_FOLDERS[k] / "preprocessing.pkl", "rb")) for k in M.POLICY_KEYS}
seq_names = {k: [f["name"] for f in schemas[k]["sequence_input"]["features"]] for k in M.POLICY_KEYS}
stat_names = {k: [f["name"] for f in schemas[k]["static_input"]["features"]] for k in M.POLICY_KEYS}
ref = "momentum"
rows = []
def check(name, cond, detail=""):
    rows.append({"check": name, "result": "PASS" if cond else "FAIL", "detail": detail})
for i, k in enumerate(M.POLICY_KEYS):
    pol, spec = bank.policies[i], M.POLICY_SPEC[k]
    check(f"{k}: policy_id is {spec['pid']}", cfgs[k].get("policy_id") == spec["pid"], str(cfgs[k].get("policy_id")))
    check(f"{k}: action mapping HOLD/BUY/SELL/EXIT", {str(a): n for a, n in cfgs[k]["action_space"].items()} == M.EXPECTED_ACTIONS, str(cfgs[k]["action_space"]))
    check(f"{k}: execution convention", cfgs[k]["execution_convention"] == cfgs[ref]["execution_convention"], cfgs[k]["execution_convention"])
    check(f"{k}: seq_len equals reference", pol.seq_len == bank.seq_len, str(pol.seq_len))
    check(f"{k}: first 30 sequence features equal SP-01 (names and order)", seq_names[k][:30] == seq_names[ref][:30])
    check(f"{k}: schema, model and preprocessing dimensions agree", len(seq_names[k]) == len(bank.columns[i]) == int(cfgs[k]["model"]["n_features"]) == len(pres[k]["feature_columns"]), f"{len(seq_names[k])} columns")
    check(f"{k}: preprocessing columns equal schema columns", list(pres[k]["feature_columns"]) == seq_names[k])
    check(f"{k}: static input dimension agrees with the saved network", len(stat_names[k]) == int(cfgs[k]["model"]["n_static"]), f"{len(stat_names[k])} columns")
    check(f"{k}: training split dates equal reference", (cfgs[k]["data"]["train_end"], cfgs[k]["data"]["val_end"]) == (cfgs[ref]["data"]["train_end"], cfgs[ref]["data"]["val_end"]),
          f"{cfgs[k]['data']['train_end']} / {cfgs[k]['data']['val_end']}")
    check(f"{k}: benchmark equals reference", cfgs[k]["data"]["benchmark"] == cfgs[ref]["data"]["benchmark"], cfgs[k]["data"]["benchmark"])
check("static prefix: SP-01 (6) is a prefix of SP-02, SP-03, SP-04", all(stat_names[k][:6] == stat_names["momentum"] for k in ("mean_reversion", "trend_following", "breakout", "conservative")))
check("static: SP-02, SP-03, SP-04 share the same 8 columns, SP-05 starts with them", stat_names["mean_reversion"] == stat_names["trend_following"] == stat_names["breakout"] == stat_names["conservative"][:8])
check("all five policies are frozen (no trainable parameters)", all(not any(p.requires_grad for p in pol.net.parameters()) for pol in bank.policies))
compat = pd.DataFrame(rows)
print(compat.groupby("result").size().to_dict())
if (compat["result"] == "FAIL").any():
    print(compat[compat["result"] == "FAIL"].to_string(index=False))
    raise RuntimeError("Policy artifacts are not mutually compatible; regenerate them with the notebooks 01-05 provided with this project.")
print("All compatibility checks passed.")
summary = pd.DataFrame({k: {"id": cfgs[k]["policy_id"], "sequence cols": len(seq_names[k]), "static cols": len(stat_names[k]), "allow_short": bank.policies[i].allow_short,
                            "temperature": round(bank.policies[i].temperature, 3), "label source": cfgs[k]["label_source"][:42]} for i, k in enumerate(M.POLICY_KEYS)}).T
print(summary.to_string())
POLICY_TRAIN_END, POLICY_VAL_END = cfgs[ref]["data"]["train_end"], cfgs[ref]["data"]["val_end"]
extra_syms = [s for s in raw_data if s not in cfgs[ref]["data"]["symbols"]]
if extra_syms:
    print(f"NOTE: symbols not used when the policies were trained: {extra_syms} (allowed, but their proposals are out-of-distribution in the policies' sense)")

## 6. Investors, observation schema, market data and shadow performance

### 6.1 Investors (SYNTHETIC unless you supply real history)
Each investor has a declared risk profile `(risk_tolerance, patience, aggressiveness)`, an investment horizon, a drawdown limit, a maximum position fraction, short authorisation and **historical behaviour statistics**. Everything is generated by a documented rule from the profile (`risk_tolerance → drawdown limit 6%…22%, position cap 25%…60%, typical position size, volatility-entry share, drawdown-exit share`; `patience → horizon and holding duration`; `aggressiveness → trade frequency`). Training uses a **population of synthetic investors** (the three archetypes plus noise); evaluation uses the three noise-free archetypes (conservative, balanced, aggressive). **No claim is made that these numbers describe a real person.** `behavior_stats_from_trades()` computes the same statistics from a real trade history for future use; behavioural *embeddings* are not used because no trained behaviour encoder exists.

### 6.2 Observation (fixed schema, 108 dimensions, only information available at the close of the current bar)
| Group | Content |
|---|---|
| **A. Market context (34)** | multi-horizon returns, MA relations, trend strength (ADX, DI), volatility and ATR, RSI, volume changes, benchmark-relative movement, volatility percentiles, drawdown, range/breakout context (z-normalised with **training-period statistics only**), plus the **past-only market regime** (one-hot: sideways, uptrend, up-weakening, downtrend, down-weakening) |
| **B. Trader behaviour (9)** | historical action frequencies (HOLD/BUY/SELL/EXIT), average holding duration, typical position size, trade frequency, reaction to volatility (share of entries in high volatility), reaction to drawdowns (share of exits in drawdown) |
| **C. Investor and position (15)** | risk tolerance, patience, aggressiveness, horizon, drawdown limit, position cap, short authorisation; current position, exposure, unrealised P&L, holding duration, distance to the next investor checkpoint (a review every `horizon` bars; used only as an input), current drawdown, recent loss, equity growth |
| **D1. Candidate policies (25)** | for each of the five policies: its four action probabilities and its confidence, given the current position and investor profile |
| **D2. Candidate-policy performance (25)** | for each policy: 20- and 60-bar return, 60-bar maximum drawdown, 20-bar Sharpe-like ratio and current position of a **shadow simulation** of that policy alone (same costs, same gate, reference investor `balanced`), computed **only from past bars** |

The realised future return of any policy is never an input, and no reward is computed from a policy's unavailable future outcome.

### 6.3 Action space
`Discrete(5)`: 0 Momentum, 1 Mean-Reversion, 2 Trend-Following, 3 Breakout, 4 Conservative. There is **no abstention action**: HOLD remains available because every policy can propose HOLD, and the risk gate can always turn a proposal into HOLD.

### 6.4 Reward (documented; every term is measurable on its own)
`reward = w_perf·(r − rf)/σ − w_dd·Δ[max(0, drawdown − tol)]/σ − w_cost·(fees+slippage)/equity/σ − w_turn·(traded/equity)·charge/σ − w_behav·behaviour_distance − rejection_penalty·[entry rejected] − forced_exit_penalty·[exit forced]`, with `σ = sigma_ref` and `tol = dd_tolerance_fraction × investor drawdown limit`.
* **Risk-adjusted performance:** the realised, cost-net step return in units of a reference volatility, with explicit penalties for **drawdown beyond the investor's tolerance** (drawdown = fall below the trailing 120-bar equity high; an all-time peak would permanently lock an account that moved to cash after a loss), **transaction costs** (an extra penalty on top of the cost already deducted from the return) and **unnecessary turnover**.
* **Behavioural consistency** is the distance between the behaviour the selector actually produced (running past-only statistics) and the investor's *historical* statistics. It is a **separate, small term** and a separately reported metric: it measures compatibility with a profile, **not** imitation of past actions and **not** profitability.
* **Rejection rule (consistent transition):** if the selected policy proposes an entry and the gate rejects it, the executed action is HOLD, the environment records the rejection and its reasons, and the reward includes `−rejection_penalty`. If the gate forces an exit, the executed action is EXIT, it is recorded, and `forced_exit_penalty` applies (default 0, because stops can be caused by the market).
* Financial performance, risk compliance (rejection rate) and behavioural consistency are reported as **three distinct quantities**.

In [ ]:
t0 = time.time()
md = M.MarketData(raw_data, bench_df, bank)                    # aligned data + every policy's own past-only features
mv = CONFIG["splits"]["meta_val_start"]
if mv is None:
    a, b = pd.Timestamp(POLICY_TRAIN_END), pd.Timestamp(POLICY_VAL_END)
    mv = str((a + (b - a) * 0.75).date())
SEGS = md.segments(POLICY_TRAIN_END, POLICY_VAL_END, mv, CONFIG["splits"]["mode"])
md.fit_normalizer(SEGS["train"][1])                            # market-context normalisation: training rows only
ECFG = M.ExecConfig(**CONFIG["execution"]); RCFG = M.RewardConfig(**CONFIG["reward"])
GATE = MetaRiskController(MetaRiskLimits(**CONFIG["risk_limits"]))
print(f"market data: {md.S} symbols x {md.T} aligned dates ({md.cal[0].date()} -> {md.cal[-1].date()}); features ready in {time.time() - t0:.0f}s")
seg_tab = pd.DataFrame({n: {"first decision": md.cal[a].date(), "last decision": md.cal[b - 1].date(), "decisions per symbol": b - a} for n, (a, b) in SEGS.items()}).T
print(seg_tab.to_string())
print(f"policies' own periods: trained <= {POLICY_TRAIN_END}, validated <= {POLICY_VAL_END} | split mode: {CONFIG['splits']['mode']}")
if CONFIG["splits"]["mode"] == "full_history":
    print("=" * 96 + "\nWARNING: full_history mode trains the selector on data the five policies were TRAINED on: their proposals and shadow performance\n"
          "are in-sample (optimistic) there. Use it only to study the pipeline, never to judge the selector.\n" + "=" * 96)
else:
    print("Selector train/validation data are AFTER the policies' training period; the selector test period is the policies' held-out test period.")
assert md.cal[SEGS["train"][1]] < md.cal[SEGS["val"][0]] and md.cal[SEGS["val"][1]] < md.cal[SEGS["test"][0]]

ARCH = M.archetype_investors()
INV_EVAL = {n: ARCH[n] for n in CONFIG["evaluation"]["archetypes"]}
REF_INV = ARCH["balanced"]
inv_tab = pd.DataFrame({n: {"risk": i.risk_tolerance, "patience": i.patience, "aggr.": i.aggressiveness, "horizon": i.horizon_bars, "dd limit": round(i.drawdown_limit, 3),
                            "max pos": round(i.max_position_fraction, 3), "shorts": i.allow_short, "SYNTHETIC": i.synthetic, **{k: round(v, 3) for k, v in i.behavior.items()}}
                        for n, i in INV_EVAL.items()}).T
print("\nEvaluation investors (SYNTHETIC, generated by documented rules; not real people):"); print(inv_tab.to_string())

t0 = time.time()
SHADOW = {name: M.run_shadow(md, bank, GATE, ECFG, SEGS[name], REF_INV) for name in SEGS}
print(f"\nshadow simulations of the five policies (past-only performance features) computed in {time.time() - t0:.0f}s:",
      {k: v.shape for k, v in SHADOW.items()})
print(f"observation: {M.OBS_DIM} dims  =  A {M.N_MARKET} + B {len(M.BEHAVIOR_KEYS)} + C {len(M.INVESTOR_FEATURES) + len(M.POSITION_FEATURES)} + D1 25 + D2 25")

### 6.5 Fast path = public path
Training calls the policies through their `predict_scaled` method on pre-scaled windows (one pass for all sleeves). The deployed interface calls the public `predict(state)` of each saved policy on RAW features. The cell below drives real simulated states (including held positions) and asserts that both paths return the same probabilities for all five policies.

In [ ]:
set_seed(CONFIG["seed"])
sim = M.SleeveSim(md, bank, GATE, ECFG, [0, 1, 2], [ARCH["aggressive"]] * 3, SEGS["val"][0] + 3)
rng_ = np.random.default_rng(1); max_diff, held_states = 0.0, 0
for step in range(60):
    sim.compute_candidates()
    for b in range(3):
        ex = sim.export_observation(b, None)
        for p in range(M.N_POLICIES):
            out = bank.policies[p].predict(ex["policy_inputs"][p])             # public interface on RAW features
            max_diff = max(max_diff, float(np.abs(out["action_probs"] - sim.probs[b, p]).max()))
        held_states += int(ex["risk_context"]["position"] != 0)
    sel = rng_.integers(0, M.N_POLICIES, size=3)
    prop, _ = sim.proposal_of(sel); sim.apply(prop)
assert max_diff < 1e-4, f"fast path and public predict() disagree: {max_diff}"
print(f"fast path vs public predict(): max |difference| = {max_diff:.2e} over 60 steps x 3 sleeves x 5 policies ({held_states} states had an open position)")

## 7. Risk-controller tests and environment checks
**Risk controller unit tests** (all assertions): the gate rejects invalid actions, enforces authorisation, blocks entries under halted conditions, forces exits at hard limits, never blocks an EXIT, and fails safe when risk data is missing. Then the **environment is checked**: Gymnasium `check_env`, equality of the single-sleeve and batched environments, and a test that the **gate overrules the selected policy** inside the simulation: whatever PPO selects, a position in breach of the drawdown limit is closed.

In [ ]:
def gate_case(name, proposed, pos=0, pnl=0.0, hold=0, dd=0.0, gross=0.0, dret=0.0, rloss=0.0, frac=0.3, feats=None, dd_limit=0.14, cap=0.5, allow_short=False, authorised=True):
    f = {"atr_pct_pctile_252": 0.4, "rv20_pctile_252": 0.4, "gap_abs_atr": 0.5, "abnormal_ret_z": 0.5, "bench_dd_60": 0.0, **(feats or {})}
    out = GATE.review_one(proposed, pos, pnl, hold, f, {"drawdown": dd, "gross_exposure": gross, "daily_return": dret, "recent_loss": rloss, "entry_fraction": frac},
                          {"drawdown_limit": dd_limit, "max_position_fraction": cap, "allow_short": allow_short, "authorised": authorised})
    return {"case": name, "proposed": out["proposed_action_name"], "final": out["final_action_name"], "approved": out["approved"], "reasons": ",".join(out["reasons"])}

cases = [
    (gate_case("normal entry", BUY), BUY, []),
    (gate_case("BUY while already long", BUY, pos=1), HOLD, ["invalid_action"]),
    (gate_case("EXIT while flat", EXIT, pos=0), HOLD, ["invalid_action"]),
    (gate_case("short not authorised", SELL), HOLD, ["short_not_authorised"]),
    (gate_case("short authorised", SELL, allow_short=True), SELL, []),
    (gate_case("trading disabled (kill switch): entry", BUY, authorised=False), HOLD, ["trading_not_authorised"]),
    (gate_case("trading disabled: EXIT still allowed", EXIT, pos=1, authorised=False), EXIT, []),
    (gate_case("hard stop-loss forces EXIT over a HOLD proposal", HOLD, pos=1, pnl=-0.07), EXIT, ["position_stop_loss"]),
    (gate_case("holding period limit forces EXIT", HOLD, pos=1, hold=130), EXIT, ["max_holding_period"]),
    (gate_case("drawdown at investor limit flattens", HOLD, pos=1, dd=0.15), EXIT, ["drawdown_flatten"]),
    (gate_case("drawdown halt blocks entries", BUY, dd=0.09), HOLD, ["drawdown_halt"]),
    (gate_case("daily loss halt blocks entries", BUY, dret=-0.035), HOLD, ["daily_loss_halt"]),
    (gate_case("recent loss halt blocks entries", BUY, rloss=0.07), HOLD, ["recent_loss_halt"]),
    (gate_case("volatility restriction", BUY, feats={"atr_pct_pctile_252": 0.95}), HOLD, ["volatility_restriction"]),
    (gate_case("abnormal gap", BUY, feats={"gap_abs_atr": 3.5}), HOLD, ["abnormal_gap_or_move"]),
    (gate_case("market stress", BUY, feats={"bench_dd_60": -0.15}), HOLD, ["market_stress"]),
    (gate_case("position larger than investor cap", BUY, frac=0.55, cap=0.5), HOLD, ["exposure_limit"]),
    (gate_case("portfolio exposure limit", BUY, gross=0.45, frac=0.3), HOLD, ["exposure_limit"]),
    (gate_case("risk data missing -> fail safe", BUY, feats={"gap_abs_atr": float("nan")}), HOLD, ["risk_data_missing"]),
    (gate_case("de-risking EXIT is approved even in a halt", EXIT, pos=1, dd=0.10, dret=-0.04), EXIT, []),
]
rows = []
for res, want_final, want_reasons in cases:
    ok = (res["final"] == ACTION_NAMES[want_final]) and all(r in res["reasons"] for r in want_reasons)
    rows.append({**res, "expected_final": ACTION_NAMES[want_final], "PASS": ok})
gate_tests = pd.DataFrame(rows)
print(gate_tests[["case", "proposed", "final", "reasons", "PASS"]].to_string(index=False))
assert gate_tests["PASS"].all(), "risk controller test failed"
print(f"\nAll {len(gate_tests)} risk-controller tests passed.")

In [ ]:
from stable_baselines3.common.env_checker import check_env

set_seed(CONFIG["seed"])
env_single = M.MetaTradingEnv(md, bank, GATE, ECFG, RCFG, SHADOW["train"], SEGS["train"], "train", CONFIG["env"]["episode_length"])
check_env(env_single, warn=True)
print("Gymnasium check_env passed (observation/action spaces, reset/step signatures, dtype and bounds).")

# single-sleeve env == batched env on the same explicit episode and actions
inv = ARCH["aggressive"]; a0 = SEGS["train"][0] + 10; a1 = a0 + 30
o1, _ = env_single.reset(options={"symbol": md.symbols[1], "investor": inv, "start": a0, "end": a1})
venv = M.BatchedMetaVecEnv(md, bank, GATE, ECFG, RCFG, SHADOW["train"], SEGS["train"], 2, "train", 30)
venv.reset()
venv.sim = M.SleeveSim(md, bank, GATE, ECFG, np.array([1, 1]), [inv, ARCH["balanced"]], a0); venv.sim.compute_candidates(); venv.end = np.array([a1, a1])
rng_ = np.random.default_rng(3); diffs = []
for i in range(29):
    a = int(rng_.integers(5))
    _, r1, _, _, _ = env_single.step(a); _, r2, _, _ = venv.step(np.array([a, a])); diffs.append(abs(r1 - float(r2[0])))
assert max(diffs) < 1e-6, f"single and batched environments disagree: {max(diffs)}"
print(f"single-sleeve and batched environments agree (max reward difference {max(diffs):.1e}).")

# the gate overrules the selected policy inside the simulation
sim = M.SleeveSim(md, bank, GATE, ECFG, [0], [ARCH["conservative"]], SEGS["val"][0] + 5)
sim.acct.shares[:] = 100.0; sim.acct.entry_px[:] = md.close[sim.k, sim.sym]; sim.acct.hold[:] = 5       # an open long position
sim.acct.peak[:] = sim.acct.eq * 1.25                                                                      # account is 20% below its peak: beyond the investor's drawdown limit
sim.compute_candidates()
for selected in range(M.N_POLICIES):
    s2 = M.SleeveSim(md, bank, GATE, ECFG, [0], [ARCH["conservative"]], SEGS["val"][0] + 5)
    s2.acct.shares[:] = 100.0; s2.acct.entry_px[:] = md.close[s2.k, s2.sym]; s2.acct.hold[:] = 5; s2.acct.peak[:] = s2.acct.eq * 1.25
    s2.compute_candidates(); prop, _ = s2.proposal_of(np.array([selected])); r = s2.apply(np.array([HOLD]))          # even a HOLD proposal
    assert int(r["final"][0]) == EXIT and "drawdown_flatten" in [REASON_NAMES[j] for j in np.where(r["reasons"][0])[0]]
print("Gate authority verified in the simulation: with the account beyond its drawdown limit, the open position is force-closed whichever policy is selected and whatever it proposes.")

## 8. PPO training
* **Algorithm:** Stable-Baselines3 PPO, `MlpPolicy`, discrete action space of five policy indices. Hyper-parameters are in `CONFIG['ppo']`.
* **Environment:** `BatchedMetaVecEnv` steps `n_envs` independent sleeves together (each with its own symbol, synthetic investor and start bar, drawn from the **selector-training** period); it is the same logic as the Gymnasium `MetaTradingEnv` (equality asserted above) but runs the five frozen policies once per step for all sleeves.
* **Episodes:** `episode_length` bars; many random (symbol, start, investor) combinations, so training covers several stocks, market periods and investor profiles.
* **Model selection and early stopping** use the **validation** period only (deterministic policy, the three evaluation archetypes × all symbols): every `eval_every_rollouts` rollouts the validation reward is computed, the best weights are kept, and training stops after `patience_evals` evaluations without improvement or at the `total_timesteps` budget. **The test period is never used for any decision.**
* **Reproducibility:** fixed seeds for Python, NumPy, PyTorch and the environments. Exact bit-for-bit reproducibility across machines is not guaranteed (GPU/CPU floating-point differences).
* **Deterministic inference:** `model.predict(obs, deterministic=True)` (argmax of the policy distribution).

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed
import stable_baselines3 as sb3

OBS_MASKS = {k: M.obs_mask(k) for k in ("full", "no_profile", "no_policy_perf")}
print("observation mask sizes (visible dims):", {k: int(v.sum()) for k, v in OBS_MASKS.items()})

class PPOSelector(M.Selector):
    """Deterministic PPO selector for evaluation (obs are already masked by run_selector)."""
    def __init__(self, name, model):
        self.model = model; self.last_probs = None
        super().__init__(name, self._fn, kind="policy", needs_all=True)
    def _fn(self, obs, sim):
        a, _ = self.model.predict(obs, deterministic=True)
        with torch.no_grad():
            self.last_probs = self.model.policy.get_distribution(torch.as_tensor(obs).to(self.model.device)).distribution.probs.cpu().numpy()
        return a

def evaluate_ppo(model, seg_name, mask_kind, rcfg=RCFG, record=False, name="PPO"):
    sel = PPOSelector(name, model)
    return M.run_selector(md, bank, GATE, ECFG, rcfg, SEGS[seg_name], SHADOW[seg_name], sel, INV_EVAL, OBS_MASKS[mask_kind], record)

class ValCallback(BaseCallback):
    def __init__(self, tag, mask_kind, rcfg, every, patience, min_evals):
        super().__init__(); self.tag, self.mask_kind, self.rcfg = tag, mask_kind, rcfg
        self.every, self.patience, self.min_evals = every, patience, min_evals
        self.n_rollouts = 0; self.best = -np.inf; self.best_state = None; self.since_best = 0; self.stop = False; self.rows = []
    def _on_step(self) -> bool:
        return not self.stop
    def _on_rollout_end(self) -> None:
        self.n_rollouts += 1
        if self.n_rollouts % self.every:
            return
        R = evaluate_ppo(self.model, "val", self.mask_kind, self.rcfg)
        v = float(R["reward"].mean()); fin = float((R["equity"][-1] - 1.0).mean())
        freq = np.bincount(R["selected"].ravel(), minlength=M.N_POLICIES) / R["selected"].size
        tr = float(np.mean([e["r"] for e in self.model.ep_info_buffer])) if len(self.model.ep_info_buffer) else float("nan")
        self.rows.append({"timesteps": int(self.num_timesteps), "val_mean_step_reward": v, "val_mean_return": fin, "train_ep_reward": tr,
                          **{f"sel_{M.POLICY_KEYS[i]}": float(freq[i]) for i in range(M.N_POLICIES)}})
        if v > self.best + 1e-9:
            self.best, self.since_best = v, 0
            self.best_state = {k: t.detach().cpu().clone() for k, t in self.model.policy.state_dict().items()}
        else:
            self.since_best += 1
        if len(self.rows) >= self.min_evals and self.since_best >= self.patience:
            self.stop = True

def train_ppo(tag, rcfg, mask_kind="full", seed=None, verbose=True):
    pc = CONFIG["ppo"]; seed = CONFIG["seed"] if seed is None else seed
    set_random_seed(seed); set_seed(seed)
    env = M.BatchedMetaVecEnv(md, bank, GATE, ECFG, rcfg, SHADOW["train"], SEGS["train"], CONFIG["env"]["n_envs"], "train",
                              CONFIG["env"]["episode_length"], OBS_MASKS[mask_kind], seed=seed)
    model = PPO("MlpPolicy", env, learning_rate=pc["learning_rate"], n_steps=pc["n_steps"], batch_size=pc["batch_size"], n_epochs=pc["n_epochs"],
                gamma=pc["gamma"], gae_lambda=pc["gae_lambda"], clip_range=pc["clip_range"], ent_coef=pc["ent_coef"], vf_coef=pc["vf_coef"],
                max_grad_norm=pc["max_grad_norm"], policy_kwargs=dict(net_arch=list(pc["net_arch"])), seed=seed, device=pc["device"], verbose=0)
    cb = ValCallback(tag, mask_kind, rcfg, pc["eval_every_rollouts"], pc["patience_evals"], pc["min_evals"])
    t0 = time.time()
    model.learn(total_timesteps=pc["total_timesteps"], callback=cb, progress_bar=False)
    if cb.best_state is None:                                        # budget ended before the first validation: evaluate once
        R = evaluate_ppo(model, "val", mask_kind, rcfg); cb.best = float(R["reward"].mean()); cb.best_state = {k: t.detach().cpu().clone() for k, t in model.policy.state_dict().items()}
    model.policy.load_state_dict(cb.best_state)                      # keep the weights with the best VALIDATION reward
    log = pd.DataFrame(cb.rows)
    if verbose:
        why = "early stopping" if cb.stop else "training budget reached"
        print(f"[{tag}] {int(model.num_timesteps):,} timesteps in {time.time() - t0:.0f}s ({why}); best validation mean step reward {cb.best:.4f} (obs mask: {mask_kind})")
    return model, log, {"timesteps": int(model.num_timesteps), "seconds": time.time() - t0, "best_val_reward": cb.best, "stopped_early": bool(cb.stop)}

set_seed(CONFIG["seed"])
MODEL_MAIN, LOG_MAIN, INFO_MAIN = train_ppo("PPO full", RCFG, "full")
print(LOG_MAIN.round(4).to_string(index=False) if len(LOG_MAIN) else "no validation checkpoints (very small budget)")

## 9. Validation
Baselines and the trained selector are evaluated on the **validation** period under identical dates, execution assumptions, initial capital and costs. This is where model choices were made (best checkpoint, early stopping); it is *not* the headline result.

**Comparators (all in the same environment, same costs, same capital, same gate unless stated):**
1. PPO Meta-RL selector (full)
2.–6. each specialised policy **alone** (a fixed selector)
7. **Regime-based heuristic selector** (rule, no learning): conservative policy in high volatility or market stress; otherwise breakout policy on a supported breakout; trend-following policy in a trend with ADX ≥ 25; mean-reversion policy in a calm range with |z-score| ≥ 1.5; otherwise momentum
8. **Buy-and-hold** with the same position size and costs, **no gate** (so it is never stopped out), plus frictionless full-capital buy-and-hold of the benchmark and of the equal-weight universe for reference
9. **Simple fixed-rule baseline:** SMA50/SMA200 golden-cross long-only (BUY when flat and SMA50 > SMA200 and close > SMA200; EXIT when SMA50 < SMA200), through the same gate

Each method is run for **every symbol and every evaluation investor**; a method's headline equity curve is the equal-weight average of its per-symbol sleeves.

In [ ]:
def perf_metrics(r, turnover=None, exposure=None) -> dict:
    r = np.asarray(r, dtype=float); n = len(r); eq = np.cumprod(1.0 + r)
    years = n / 252.0; sd = r.std(ddof=1) if n > 1 else np.nan
    ex = r - CONFIG["reward"]["rf_annual"] / 252.0
    dd_sd = np.sqrt((np.minimum(ex, 0.0) ** 2).mean()) if n else np.nan
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    dd = eq / peak - 1.0; mdd = float(dd.min())
    cagr = float(eq[-1] ** (1 / years) - 1) if years > 0 and eq[-1] > 0 else float("nan")
    var5 = float(np.percentile(r, 5)) if n else float("nan")
    out = {"n_days": int(n), "total_return": float(eq[-1] - 1.0), "cagr": cagr,
           "ann_vol": float(sd * np.sqrt(252)) if sd == sd else float("nan"),
           "sharpe": float(ex.mean() / sd * np.sqrt(252)) if sd and sd > 0 else float("nan"),
           "sortino": float(ex.mean() / dd_sd * np.sqrt(252)) if dd_sd and dd_sd > 0 else float("nan"),
           "max_drawdown": mdd, "calmar": float(cagr / abs(mdd)) if mdd < 0 and cagr == cagr else float("nan"),
           "downside_deviation_ann": float(dd_sd * np.sqrt(252)) if dd_sd == dd_sd else float("nan"),
           "var_95_daily": var5, "cvar_95_daily": float(r[r <= var5].mean()) if n and (r <= var5).any() else float("nan"),
           "worst_day": float(r.min()) if n else float("nan"), "ulcer_index": float(np.sqrt(np.mean((100 * dd) ** 2))),
           "share_days_underwater": float((dd < 0).mean())}
    out["annualized_turnover"] = float(np.mean(turnover) * 252.0) if turnover is not None else 0.0
    out["avg_gross_exposure"] = float(np.mean(exposure)) if exposure is not None else 1.0
    out["max_gross_exposure"] = float(np.max(exposure)) if exposure is not None else 1.0
    return out

def heuristic_fn(obs, sim):
    volp = np.maximum(sim.raw("atr_pct_pctile_252"), sim.raw("rv20_pctile_252")); stress = sim.raw("bench_dd_60") <= -0.10
    adx, z = sim.raw("adx_14"), sim.raw("zscore_20")
    supported_break = (sim.raw("brk_vol_surge_20") >= 1.3) & ((sim.raw("brk_dist_high_20") > 0) | (sim.raw("brk_dist_low_20") < 0))
    with np.errstate(invalid="ignore"):
        sel = np.where(adx >= 25, 2, np.where((np.abs(z) >= 1.5) & (adx < 20), 1, 0))      # trend -> 2, calm range extreme -> 1, else momentum
        sel = np.where(supported_break, 3, sel)                                              # supported breakout -> 3
        sel = np.where((volp >= 0.85) | stress, 4, sel)                                      # high volatility / market stress -> conservative (highest precedence)
    return sel.astype(int)

def buyhold_fn(obs, sim):
    return np.where(sim.context()["pos"] == 0, BUY, HOLD)

def goldencross_fn(obs, sim):
    pos = sim.context()["pos"]; spread, c200 = sim.raw("sma50_sma200"), sim.raw("close_sma200")
    with np.errstate(invalid="ignore"):
        return np.where((pos == 0) & (spread > 0) & (c200 > 0), BUY, np.where((pos == 1) & (spread < 0), EXIT, HOLD))

NAME_PPO = "PPO Meta-RL selector (full)"
NAME_NOPROF = "PPO ablation: no trader-profile inputs"
NAME_NOPERF = "PPO ablation: no candidate-policy performance"
NAME_HEUR = "Regime-based heuristic selector"
NAME_BH = "Buy & hold (same size/costs, no gate)"
NAME_GC = "Fixed rule: SMA50/200 golden cross (gated)"
POLICY_ALONE = [f"{n} alone" for n in M.POLICY_NAMES]
BASELINES = {f"{n} alone": M.Selector(f"{n} alone", (lambda o, s, p=i: np.full(s.B, p)), needs_all=False) for i, n in enumerate(M.POLICY_NAMES)}
BASELINES[NAME_HEUR] = M.Selector(NAME_HEUR, heuristic_fn, needs_all=False)
BASELINES[NAME_BH] = M.Selector(NAME_BH, buyhold_fn, kind="direct", needs_all=False, gate_on=False)
BASELINES[NAME_GC] = M.Selector(NAME_GC, goldencross_fn, kind="direct", needs_all=False, gate_on=True)
ARCH_NAMES = list(INV_EVAL)

def run_baselines(seg_name):
    t0 = time.time(); out = {}
    for n, sel in BASELINES.items():
        out[n] = M.run_selector(md, bank, GATE, ECFG, RCFG, SEGS[seg_name], SHADOW[seg_name], sel, INV_EVAL)
    print(f"[{seg_name}] {len(out)} baselines evaluated in {time.time() - t0:.0f}s")
    return out

def portfolio_returns(R, arch=None):
    m = np.ones(R["inv"].shape, bool) if arch in (None, "ALL") else (R["inv"] == arch)
    return R["ret"][:, m].mean(axis=1)                         # equal-weight over the sleeves (symbols, and investors if ALL)

def summarize(R, arch=None) -> dict:
    m = np.ones(R["inv"].shape, bool) if arch in (None, "ALL") else (R["inv"] == arch)
    r = portfolio_returns(R, arch)
    out = perf_metrics(r, R["traded"][:, m].mean(axis=1), R["exposure"][:, m].mean(axis=1))
    prop_entry = (R["proposed"][:, m] == BUY) | (R["proposed"][:, m] == SELL)
    out["trade_count"] = int(R["entered"][:, m].sum())
    out["trades_per_sleeve"] = out["trade_count"] / max(int(m.sum()), 1)
    out["rejection_rate"] = float(R["blocked"][:, m].sum() / max(prop_entry.sum(), 1))
    out["override_rate"] = float((R["final"][:, m] != R["proposed"][:, m]).mean())
    out["forced_exit_rate"] = float(R["forced"][:, m].mean())
    out["behaviour_distance"] = float(R["beh"][:, m].mean())
    out["mean_step_reward"] = float(R["reward"][:, m].mean())
    return out

def table_for(runs, arch=None):
    return pd.DataFrame({n: summarize(R, arch) for n, R in runs.items()}).T

def reference_rows(seg_name):
    k0, k1 = SEGS[seg_name]
    spy = md.bench_close[k0 + 1:k1 + 1] / md.bench_close[k0:k1] - 1.0
    ew = (md.close[k0 + 1:k1 + 1] / md.close[k0:k1] - 1.0).mean(axis=1)
    return {f"{CONFIG['benchmark']} buy & hold (frictionless, full capital)": spy, "Equal-weight universe buy & hold (frictionless)": ew}

def show_cols(t):
    return t[["total_return", "sharpe", "sortino", "max_drawdown", "annualized_turnover", "trade_count", "avg_gross_exposure", "rejection_rate", "behaviour_distance"]].round(3)

RUNS = {"val": run_baselines("val")}
RUNS["val"][NAME_PPO] = evaluate_ppo(MODEL_MAIN, "val", "full", name=NAME_PPO)
VAL_TABLE = {a: table_for(RUNS["val"], a) for a in ARCH_NAMES + ["ALL"]}
print("\nVALIDATION period, all investors pooled (equal-weight over symbols and investors):")
print(show_cols(VAL_TABLE["ALL"]).to_string())

## 10. Held-out test, ablations and reward sensitivity
The **test period** (the policies' own held-out period) is evaluated once for every method, with the model weights fixed by validation. Two ablations then retrain the selector with the same budget and the same validation-based selection:
* **no trader-profile inputs:** group B (behaviour history) and the investor profile/limit inputs are hidden from the selector (the environment, gate and reward still use the investor);
* **no candidate-policy performance:** group D2 (shadow performance of each policy) is hidden; the selector still sees the policies' current proposals.

If these ablations do as well as the full selector, personalisation or dynamic selection is **not** adding value in this experiment, and the results below will say so.

**Reward sensitivity:** the selector is retrained with changed reward weights (same budget, same selection rule) and evaluated with the same financial metrics. Reward *values* are not comparable across variants; the financial, risk-compliance and consistency metrics are.

In [ ]:
RUNS["test"] = run_baselines("test")
t0 = time.time()
RUNS["test"][NAME_PPO] = evaluate_ppo(MODEL_MAIN, "test", "full", record=True, name=NAME_PPO)
print(f"[test] PPO selector evaluated in {time.time() - t0:.0f}s")

MODELS = {NAME_PPO: (MODEL_MAIN, "full", INFO_MAIN, LOG_MAIN)}
if CONFIG["ablations"]["run"]:
    for name, kind in ((NAME_NOPROF, "no_profile"), (NAME_NOPERF, "no_policy_perf")):
        m_, log_, info_ = train_ppo(name, RCFG, kind)
        MODELS[name] = (m_, kind, info_, log_)
        RUNS["val"][name] = evaluate_ppo(m_, "val", kind, name=name)
        RUNS["test"][name] = evaluate_ppo(m_, "test", kind, name=name)

SENS = {}
if CONFIG["reward_sensitivity"]["run"]:
    for vname, over in CONFIG["reward_sensitivity"]["variants"].items():
        rc_v = M.RewardConfig(**{**CONFIG["reward"], **over})
        m_, log_, info_ = train_ppo(f"reward variant: {vname}", rc_v, "full")
        SENS[vname] = {"overrides": over, "model": m_, "info": info_, "log": log_}
        RUNS["test"][f"PPO reward variant: {vname}"] = evaluate_ppo(m_, "test", "full", rcfg=RCFG, name=vname)

TEST_TABLE = {a: table_for(RUNS["test"], a) for a in ARCH_NAMES + ["ALL"]}
MAIN_METHODS = [NAME_PPO] + POLICY_ALONE + [NAME_HEUR, NAME_BH, NAME_GC]
print("\nHELD-OUT TEST, all investors pooled (equal-weight over symbols and investors; costs, slippage and gate included):")
print(show_cols(TEST_TABLE["ALL"].loc[MAIN_METHODS]).to_string())
refs = {k: perf_metrics(v) for k, v in reference_rows("test").items()}
print("\nreference series (frictionless, no sizing, no gate):")
print(pd.DataFrame(refs).T[["total_return", "sharpe", "sortino", "max_drawdown"]].round(3).to_string())

## 11. Analysis
Financial performance, risk compliance and behavioural consistency are shown separately.

* **By investor risk profile:** the same table per archetype.
* **By market regime:** daily returns grouped by the **benchmark regime known at the previous close** (past-only).
* **Selection behaviour:** how often PPO chooses each policy, by investor and by regime.
* **Risk gate:** rejection rate (rejected entries / proposed entries), override rate and reasons.
* **Behavioural consistency:** distance between the behaviour a method produced and the investor's historical statistics (0 = identical). It is **not** a quality score.
* **Ablations and reward sensitivity.**
* **Is the selector better?** A paired block-bootstrap on daily returns compares PPO with each baseline; the verdict is computed from the results and states plainly when the evidence does not support superiority.

In [ ]:
print("=== BY INVESTOR RISK PROFILE (TEST) ===")
for a in ARCH_NAMES:
    print(f"\n-- {a} --"); print(show_cols(TEST_TABLE[a].loc[MAIN_METHODS]).to_string())

# ---- by market regime (benchmark regime known at the previous close) -----------------------------------------------------
k0, k1 = SEGS["test"]; rg = md.bench_regime[k0:k1]
def by_regime(runs, names, arch="ALL"):
    rows = []
    for n in names:
        r = portfolio_returns(runs[n], arch)
        for code_, rname in list(enumerate(M.REGIME_NAMES)):
            m = rg == code_
            if m.sum() >= 5:
                x = r[m]; sd = x.std(ddof=1)
                rows.append({"method": n, "regime": rname, "days": int(m.sum()), "compounded_return": float(np.prod(1 + x) - 1),
                             "ann_return": float(x.mean() * 252), "sharpe": float(x.mean() / sd * np.sqrt(252)) if sd > 0 else np.nan})
    return pd.DataFrame(rows)
REGIME_PERF = by_regime(RUNS["test"], MAIN_METHODS)
print("\n=== BY MARKET REGIME (TEST; benchmark regime known at the previous close; days per regime in the 'days' column) ===")
print("share of test days by regime:", pd.Series(rg).map(lambda v: M.REGIME_NAMES[v] if v >= 0 else "unknown").value_counts(normalize=True).round(3).to_dict())
print(REGIME_PERF.pivot(index="method", columns="regime", values="compounded_return").reindex(MAIN_METHODS).round(3).to_string())
print("Sharpe by regime:"); print(REGIME_PERF.pivot(index="method", columns="regime", values="sharpe").reindex(MAIN_METHODS).round(2).to_string())

# ---- policy-selection frequency ---------------------------------------------------------------------------------------
Rp = RUNS["test"][NAME_PPO]
SEL_FREQ = {}
for a in ARCH_NAMES + ["ALL"]:
    m = np.ones(Rp["inv"].shape, bool) if a == "ALL" else (Rp["inv"] == a)
    SEL_FREQ[a] = np.bincount(Rp["selected"][:, m].ravel(), minlength=M.N_POLICIES) / Rp["selected"][:, m].size
SEL_TAB = pd.DataFrame(SEL_FREQ, index=M.POLICY_NAMES).T
print("\n=== POLICY-SELECTION FREQUENCY of the PPO selector (TEST) ===")
print(SEL_TAB.round(3).to_string())
sel_reg = {}
for code_, rname in enumerate(M.REGIME_NAMES):
    m = rg == code_
    if m.sum() >= 5:
        sel_reg[rname] = np.bincount(Rp["selected"][m].ravel(), minlength=M.N_POLICIES) / Rp["selected"][m].size
SEL_REG_TAB = pd.DataFrame(sel_reg, index=M.POLICY_NAMES).T
print("by SPY regime:"); print(SEL_REG_TAB.round(3).to_string())
print("heuristic selector's frequencies for comparison:", dict(zip(M.POLICY_NAMES, np.round(np.bincount(RUNS['test'][NAME_HEUR]['selected'].ravel(), minlength=5) / RUNS['test'][NAME_HEUR]['selected'].size, 3))))
if SEL_TAB.loc["ALL"].max() > 0.9:
    print(f"NOTE: PPO selects '{M.POLICY_NAMES[int(SEL_TAB.loc['ALL'].values.argmax())]}' {SEL_TAB.loc['ALL'].max():.0%} of the time: it has largely collapsed to one policy; dynamic selection is not doing much work.")

# ---- risk gate ----------------------------------------------------------------------------------------------------------
reason_counts = {}
for r in Rp["reasons"]:
    for j, n in enumerate(REASON_NAMES):
        reason_counts[n] = reason_counts.get(n, 0) + int(r[:, j].sum())
GATE_TAB = pd.Series(reason_counts); GATE_TAB = GATE_TAB[GATE_TAB > 0].sort_values(ascending=False)
print("\n=== RISK GATE (PPO selector, TEST) ===")
print(table_for({NAME_PPO: Rp}, "ALL")[["rejection_rate", "override_rate", "forced_exit_rate"]].round(4).to_string())
print("rejections/overrides by reason (count of sleeve-days):", GATE_TAB.to_dict() or "none")
print("rejection rate by method:"); print(TEST_TABLE["ALL"].loc[MAIN_METHODS][["rejection_rate", "override_rate", "forced_exit_rate"]].round(4).to_string())

# ---- behavioural consistency (SEPARATE from performance) ---------------------------------------------------------------
BEH = pd.DataFrame({a: TEST_TABLE[a].loc[MAIN_METHODS]["behaviour_distance"] for a in ARCH_NAMES + ["ALL"]})
print("\n=== BEHAVIOURAL CONSISTENCY (TEST): distance to the investor's historical behaviour; 0 = identical; NOT a quality or profit measure ===")
print(BEH.round(4).to_string())
print(f"correlation between behavioural distance and Sharpe across methods (all investors): {np.corrcoef(BEH['ALL'].values, TEST_TABLE['ALL'].loc[MAIN_METHODS]['sharpe'].values.astype(float))[0, 1]:+.2f} "
      f"(a weak relation means consistency and performance are genuinely different quantities)")

In [ ]:
# ---- ablations and reward sensitivity -----------------------------------------------------------------------------------
ABL_NAMES = [n for n in (NAME_PPO, NAME_NOPROF, NAME_NOPERF) if n in RUNS["test"]]
print("=== ABLATIONS (TEST, all investors pooled; same budget, same validation-based selection) ===")
print(show_cols(TEST_TABLE["ALL"].loc[ABL_NAMES]).to_string())
print("\nselection frequencies of the ablation models:")
for n in ABL_NAMES:
    print(f"  {n}: " + ", ".join(f"{M.POLICY_KEYS[i]} {f:.2f}" for i, f in enumerate(np.bincount(RUNS['test'][n]['selected'].ravel(), minlength=5) / RUNS['test'][n]['selected'].size)))
if len(ABL_NAMES) == 3:
    full_s = TEST_TABLE["ALL"].loc[NAME_PPO, "sharpe"]
    for n in (NAME_NOPROF, NAME_NOPERF):
        d = full_s - TEST_TABLE["ALL"].loc[n, "sharpe"]
        print(f"Sharpe(full) - Sharpe({n}) = {d:+.3f}  ->  " + ("the removed inputs helped in this run" if d > 0.1 else "no clear benefit from the removed inputs in this run (within noise of a single training run)"))

SENS_NAMES = [f"PPO reward variant: {v}" for v in SENS]
if SENS:
    SENS_TAB = show_cols(TEST_TABLE["ALL"].loc[[NAME_PPO] + SENS_NAMES]).copy()
    SENS_TAB.insert(0, "reward overrides", ["(baseline weights)"] + [str(SENS[v]["overrides"]) for v in SENS])
    SENS_TAB["selection: most used policy"] = [M.POLICY_KEYS[int(np.bincount(RUNS['test'][n]['selected'].ravel(), minlength=5).argmax())] for n in [NAME_PPO] + SENS_NAMES]
    print("\n=== REWARD SENSITIVITY (TEST, all investors pooled) ===")
    print(SENS_TAB.to_string())
    print("Differences between variants below the single-run noise level should not be over-interpreted.")
    for v in SENS:
        a_, b_ = TEST_TABLE["ALL"].loc[NAME_PPO], TEST_TABLE["ALL"].loc[f"PPO reward variant: {v}"]
        if np.allclose(a_[["total_return", "sharpe", "trade_count"]].astype(float).values, b_[["total_return", "sharpe", "trade_count"]].astype(float).values, atol=1e-12):
            print(f"NOTE: variant '{v}' produced results identical to the baseline: the changed reward term was (almost) never active in the training episodes, so the learning signal was unchanged.")

# ---- is the selector better? paired block bootstrap on daily portfolio returns ---------------------------------------------
def paired_bootstrap(ra, rb, reps, block, seed=0):
    rng = np.random.default_rng(seed); n = len(ra); nb = int(np.ceil(n / block))
    d_mean, d_sharpe = [], []
    for _ in range(reps):
        st = rng.integers(0, n - block + 1, size=nb); idx = (st[:, None] + np.arange(block)[None, :]).ravel()[:n]
        a, b = ra[idx], rb[idx]
        sa, sb = a.std(ddof=1), b.std(ddof=1)
        d_mean.append(a.mean() - b.mean())
        d_sharpe.append((a.mean() / sa - b.mean() / sb) * np.sqrt(252) if sa > 0 and sb > 0 else 0.0)
    q = lambda x: [float(np.percentile(x, 5)), float(np.percentile(x, 95))]
    return {"mean_daily_diff_ci90": q(d_mean), "sharpe_diff_ci90": q(d_sharpe), "p_sharpe_diff_gt0": float(np.mean(np.array(d_sharpe) > 0))}

BOOT = {}
r_ppo = portfolio_returns(RUNS["test"][NAME_PPO], "ALL")
others = POLICY_ALONE + [NAME_HEUR, NAME_BH, NAME_GC]
for n in others:
    BOOT[n] = paired_bootstrap(r_ppo, portfolio_returns(RUNS["test"][n], "ALL"), CONFIG["evaluation"]["bootstrap_reps"], CONFIG["evaluation"]["bootstrap_block"])
BOOT_TAB = pd.DataFrame({n: {"Sharpe(PPO)-Sharpe(other) 5th pct": v["sharpe_diff_ci90"][0], "95th pct": v["sharpe_diff_ci90"][1], "P(diff>0)": v["p_sharpe_diff_gt0"]} for n, v in BOOT.items()}).T
print("\n=== PAIRED BLOCK BOOTSTRAP: PPO selector minus each baseline (TEST, all investors pooled, 90% interval of the annualised Sharpe difference) ===")
print(BOOT_TAB.round(3).to_string())
ppo_sharpe = TEST_TABLE["ALL"].loc[NAME_PPO, "sharpe"]
best_base = TEST_TABLE["ALL"].loc[others, "sharpe"].astype(float).idxmax()
beats_all = all(v["sharpe_diff_ci90"][0] > 0 for v in BOOT.values())
beats_best = BOOT[best_base]["sharpe_diff_ci90"][0] > 0
if beats_all:
    VERDICT = (f"SUPPORTED on this test period: PPO's Sharpe ({ppo_sharpe:.2f}) exceeds every baseline with the 90% bootstrap interval of the difference above zero. "
               "This is one period, one symbol list and one training run; it is not evidence of live profitability.")
elif beats_best:
    VERDICT = (f"PARTLY SUPPORTED: PPO beats the best baseline ({best_base}) with a positive 90% interval, but not every baseline. Treat with caution (single run, one period).")
else:
    VERDICT = (f"NOT SUPPORTED: PPO's Sharpe is {ppo_sharpe:.2f}; the best baseline ({best_base}) has {TEST_TABLE['ALL'].loc[best_base, 'sharpe']:.2f}, and the 90% bootstrap interval of the "
               "difference includes zero or is negative for at least one baseline. This experiment does not show that the Meta-RL selector is superior.")
print("\nVERDICT:", VERDICT)

## 12. Plots

In [ ]:
FIG_DIR = ART_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
dates_test = md.cal[SEGS["test"][0] + 1:SEGS["test"][1] + 1]
COL = {NAME_PPO: "tab:red", NAME_HEUR: "tab:green", NAME_BH: "black", NAME_GC: "tab:brown"}

# learning curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if len(LOG_MAIN):
    ax[0].plot(LOG_MAIN["timesteps"], LOG_MAIN["val_mean_step_reward"], label="validation mean step reward")
    ax[0].plot(LOG_MAIN["timesteps"], LOG_MAIN["train_ep_reward"] / CONFIG["env"]["episode_length"], label="training episode reward / length")
    ax[0].set_xlabel("timesteps"); ax[0].set_title("PPO learning curves"); ax[0].legend()
    LOG_MAIN.set_index("timesteps")[[f"sel_{k}" for k in M.POLICY_KEYS]].plot(kind="area", ax=ax[1], stacked=True, alpha=0.7)
    ax[1].set_title("validation: selection frequency during training"); ax[1].set_ylim(0, 1); ax[1].legend(fontsize=6)
plt.tight_layout(); plt.savefig(FIG_DIR / "learning_curves.png", dpi=130); plt.show()

# equity curves
fig, ax = plt.subplots(3, 1, figsize=(11, 12), sharex=True, gridspec_kw={"height_ratios": [3, 1.6, 1.2]})
for n in MAIN_METHODS:
    r = portfolio_returns(RUNS["test"][n], "ALL"); eq = np.cumprod(1 + r)
    ax[0].plot(dates_test, eq, label=n, lw=2.4 if n == NAME_PPO else 1.0, color=COL.get(n))
    if n in (NAME_PPO, NAME_BH, NAME_HEUR):
        ax[1].plot(dates_test, eq / np.maximum.accumulate(eq) - 1, label=n, color=COL.get(n))
ax[0].set_title("HELD-OUT TEST: equal-weight over symbols and investors, after costs"); ax[0].set_ylabel("growth of 1.0"); ax[0].legend(fontsize=6, ncol=2)
ax[1].set_ylabel("drawdown"); ax[1].legend(fontsize=7)
ax[2].plot(dates_test, RUNS["test"][NAME_PPO]["exposure"].mean(axis=1), color="tab:red"); ax[2].set_ylabel("PPO gross exposure")
plt.tight_layout(); plt.savefig(FIG_DIR / "equity_test.png", dpi=130); plt.show()

# selection frequency and gate
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
SEL_TAB.plot(kind="bar", stacked=True, ax=ax[0]); ax[0].set_title("PPO selection frequency by investor"); ax[0].legend(fontsize=6); plt.setp(ax[0].get_xticklabels(), rotation=0)
SEL_REG_TAB.plot(kind="bar", stacked=True, ax=ax[1], legend=False); ax[1].set_title("PPO selection frequency by SPY regime"); plt.setp(ax[1].get_xticklabels(), rotation=20, fontsize=7)
if len(GATE_TAB):
    GATE_TAB.plot(kind="barh", ax=ax[2], color="tab:red"); ax[2].set_title("risk gate: rejections/overrides by reason (PPO, test)")
plt.tight_layout(); plt.savefig(FIG_DIR / "selection_and_gate.png", dpi=130); plt.show()

# regimes and investors
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
REGIME_PERF.pivot(index="regime", columns="method", values="compounded_return")[[n for n in (NAME_PPO, NAME_HEUR, NAME_BH) if n in MAIN_METHODS]].plot(kind="bar", ax=ax[0])
ax[0].axhline(0, color="k", lw=0.8); ax[0].set_title("TEST: compounded return by SPY regime"); plt.setp(ax[0].get_xticklabels(), rotation=15, fontsize=7); ax[0].legend(fontsize=6)
pd.DataFrame({a: TEST_TABLE[a].loc[[NAME_PPO, NAME_HEUR, NAME_BH] + POLICY_ALONE]["sharpe"] for a in ARCH_NAMES}).astype(float).plot(kind="bar", ax=ax[1])
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("TEST: Sharpe by investor profile"); plt.setp(ax[1].get_xticklabels(), rotation=25, ha="right", fontsize=6)
plt.tight_layout(); plt.savefig(FIG_DIR / "regime_and_investor.png", dpi=130); plt.show()

# ablations, sensitivity, consistency vs performance
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
TEST_TABLE["ALL"].loc[ABL_NAMES][["sharpe"]].astype(float).plot(kind="bar", ax=ax[0], legend=False); ax[0].set_title("ablations: test Sharpe"); plt.setp(ax[0].get_xticklabels(), rotation=15, ha="right", fontsize=6)
if SENS:
    TEST_TABLE["ALL"].loc[[NAME_PPO] + SENS_NAMES][["sharpe"]].astype(float).plot(kind="bar", ax=ax[1], legend=False); ax[1].set_title("reward sensitivity: test Sharpe"); plt.setp(ax[1].get_xticklabels(), rotation=15, ha="right", fontsize=6)
ax[2].scatter(BEH["ALL"].values, TEST_TABLE["ALL"].loc[MAIN_METHODS]["sharpe"].astype(float).values)
for n, x, y in zip(MAIN_METHODS, BEH["ALL"].values, TEST_TABLE["ALL"].loc[MAIN_METHODS]["sharpe"].astype(float).values):
    ax[2].annotate(n[:14], (x, y), fontsize=6)
ax[2].set_xlabel("behavioural distance (lower = more consistent)"); ax[2].set_ylabel("test Sharpe"); ax[2].set_title("consistency vs performance are different things")
plt.tight_layout(); plt.savefig(FIG_DIR / "ablations_sensitivity_consistency.png", dpi=130); plt.show()

## 13. Export
| File | Content |
|---|---|
| `meta_rl_selector.zip` | the trained Stable-Baselines3 PPO checkpoint (policy network weights; best validation checkpoint) |
| `meta_rl_config.json` | PPO hyper-parameters, seeds, reward weights, split dates, policy registry (ids, names, SHA-256 of each artifact, dimensions), observation mask, intended use, synthetic-investor statement |
| `meta_rl_observation_schema.json` | every observation dimension by name and group, normalisation statistics, scaling constants |
| `risk_controller_config.json` | the global limits, the reason codes, the investor-limit rules and the authority statement |
| `evaluation_metrics.json` | validation/test tables, per-investor and per-regime results, selection frequencies, gate statistics, behavioural consistency, ablations, reward sensitivity, bootstrap, verdict |
| `policy_selection_log.csv` | every PPO decision on the test period: date, symbol, investor, selected policy and probabilities, proposal, final action, gate decision and reasons, reward |
| `equity_curve.csv` | equity of every method on the test period per investor profile |
| `meta_rl_core.py`, `meta_rl_risk.py` | the exact code needed to load and use the selector |

In [ ]:
def to_py(o):
    if isinstance(o, pd.DataFrame): return to_py(o.reset_index().to_dict(orient="records"))
    if isinstance(o, pd.Series): return to_py(o.to_dict())
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if hasattr(o, "isoformat"): return str(o)
    return o

def sha256(path):
    h = hashlib.sha256(); h.update(Path(path).read_bytes()); return h.hexdigest()

# 1) the PPO checkpoint
MODEL_MAIN.save(str(ART_DIR / "meta_rl_selector"))                       # writes meta_rl_selector.zip

# 2) config files
policy_registry = [{"policy_id": i, "key": k, "name": M.POLICY_NAMES[i], "artifact": M.POLICY_SPEC[k]["pt"], "sp_id": M.POLICY_SPEC[k]["pid"],
                    "sha256": sha256(POLICY_FOLDERS[k] / M.POLICY_SPEC[k]["pt"]), "n_sequence_features": len(bank.columns[i]),
                    "n_static_features": len(stat_names[k]), "seq_len": bank.policies[i].seq_len, "allow_short": bank.policies[i].allow_short,
                    "confidence_temperature": float(bank.policies[i].temperature), "required_files": REQUIRED[k]} for i, k in enumerate(M.POLICY_KEYS)]
meta_config = {
    "model_name": "Personalized Meta-RL Strategy Selector", "algorithm": f"PPO (stable-baselines3 {sb3.__version__})", "artifact": "meta_rl_selector.zip",
    "policy_artifact_root": str(POLICY_ROOT.resolve()), "policies": policy_registry, "observation_mask": "full", "obs_dim": M.OBS_DIM,
    "action_space": {str(i): f"select {n}" for i, n in enumerate(M.POLICY_NAMES)}, "ppo": CONFIG["ppo"], "env": CONFIG["env"], "seed": CONFIG["seed"],
    "reward": RCFG.to_dict(), "execution": CONFIG["execution"],
    "splits": {"mode": CONFIG["splits"]["mode"], **{n: {"first_decision": str(md.cal[a].date()), "last_decision": str(md.cal[b - 1].date())} for n, (a, b) in SEGS.items()},
               "policy_train_end": POLICY_TRAIN_END, "policy_val_end": POLICY_VAL_END},
    "data": {"symbols": md.symbols, "benchmark": CONFIG["benchmark"], "source": "yfinance daily adjusted OHLCV"},
    "training": {"timesteps": INFO_MAIN["timesteps"], "seconds": INFO_MAIN["seconds"], "best_validation_step_reward": INFO_MAIN["best_val_reward"], "stopped_early": INFO_MAIN["stopped_early"]},
    "investors": {"synthetic": True, "statement": "All investor profiles and behaviour histories used for training and evaluation are SYNTHETIC, generated by documented rules; "
                                                  "no real individual's behaviour was learned. Behavioural embeddings are not used (no trained encoder exists)."},
    "intended_use": "research prototype for offline evaluation and paper trading only; not an unrestricted live-trading system",
    "verdict_on_test": VERDICT,
}
(ART_DIR / "meta_rl_config.json").write_text(json.dumps(to_py(meta_config), indent=2))

groups = {"A_market_context": (0, M.N_MARKET), "B_trader_behavior": (M.N_MARKET, M.N_MARKET + 9), "C_investor_and_position": (M.N_MARKET + 9, M.N_MARKET + 24),
          "D1_candidate_policy_proposals": (M.N_MARKET + 24, M.N_MARKET + 49), "D2_candidate_policy_recent_performance": (M.N_MARKET + 49, M.OBS_DIM)}
obs_schema = {"obs_dim": M.OBS_DIM, "dtype": "float32", "bounds": [-10.0, 10.0], "decision_time": "all values use only information available at the close of the current bar",
              "groups": {g: {"start": a, "end": b, "names": M.OBS_NAMES[a:b]} for g, (a, b) in groups.items()},
              "normalisation": {"market_context": {"features": M.MARKET_FEATURES, "mean": md.mean.tolist(), "std": md.std.tolist(), "clip": 5.0,
                                                     "fit_rows": f"decision bars from the first valid bar to {md.cal[SEGS['train'][1]].date()} (training data only)"},
                                "regime_one_hot": M.REGIME_NAMES, "investor_features": M.INVESTOR_FEATURES, "position_features": M.POSITION_FEATURES,
                                "behavior_keys": M.BEHAVIOR_KEYS, "shadow_metrics": M.SHADOW_METRICS,
                                "constants": {"horizon_norm": "horizon_bars / 120", "drawdown_limit_norm": "limit / 0.25", "pnl": "clip(pnl/0.1, +-3)",
                                              "holding": "min(bars, 60)/20", "drawdown": "clip(dd/0.1, 0, 3)", "recent_loss": "clip(loss/0.05, 0, 3)",
                                              "equity_growth": "clip((equity/capital-1)/0.2, +-3)", "shadow": "ret20/0.1, ret60/0.2, maxdd60/0.1 (clipped), sharpe20/3, position"}},
              "ablation_masks": {k: int(v.sum()) for k, v in OBS_MASKS.items()}, "this_model_mask": "full"}
(ART_DIR / "meta_rl_observation_schema.json").write_text(json.dumps(to_py(obs_schema), indent=2))

risk_cfg = {"limits": GATE.limits.to_dict(), "reason_codes": REASON_NAMES, "forced_exit_reasons": FORCED_EXIT_REASONS,
            "investor_limits": {"drawdown_limit": "0.06 + 0.16 * risk_tolerance (flatten at the limit, entries halted at drawdown_halt_fraction of it)",
                                "max_position_fraction": "0.25 + 0.35 * risk_tolerance", "allow_short": "risk_tolerance > 0.65 and aggressiveness > 0.6", "authorised": True},
            "authority": "The controller can reject any proposal regardless of the selector's choice or confidence; the selector never bypasses it. EXIT proposals are never blocked.",
            "deterministic": True, "learned_parameters": 0}
(ART_DIR / "risk_controller_config.json").write_text(json.dumps(to_py(risk_cfg), indent=2))

# 3) evaluation metrics
evaluation = {
    "disclaimer": "Research prototype for offline evaluation and paper trading only. Simulated results do not imply live profitability.",
    "investors": "SYNTHETIC profiles; behavioural statistics are generated by documented rules",
    "verdict": VERDICT, "splits": meta_config["splits"], "training_info": {n: v[2] for n, v in MODELS.items()},
    "validation": {a: VAL_TABLE[a] for a in VAL_TABLE}, "test": {a: TEST_TABLE[a] for a in TEST_TABLE},
    "reference_series_test": refs, "by_regime_test": REGIME_PERF, "selection_frequency_test": SEL_TAB, "selection_frequency_by_regime_test": SEL_REG_TAB,
    "gate_rejections_by_reason_ppo_test": GATE_TAB, "behavioural_consistency_test": BEH, "bootstrap_ppo_minus_baseline": BOOT,
    "ablations": TEST_TABLE["ALL"].loc[ABL_NAMES], "reward_sensitivity": ({v: {"overrides": SENS[v]["overrides"], "training": SENS[v]["info"]} for v in SENS}),
    "reward_sensitivity_test_metrics": (TEST_TABLE["ALL"].loc[[NAME_PPO] + SENS_NAMES] if SENS else {}),
    "risk_controller_tests_passed": int(gate_tests["PASS"].sum()), "policy_compatibility_checks": compat,
    "training_log_main": LOG_MAIN,
}
(ART_DIR / "evaluation_metrics.json").write_text(json.dumps(to_py(evaluation), indent=2))

# 4) selection log (PPO, test) and equity curves
Rp = RUNS["test"][NAME_PPO]; K, B = Rp["selected"].shape; rows_ = []
dts = md.cal[SEGS["test"][0]:SEGS["test"][1]]
reason_str = [[";".join(REASON_NAMES[j] for j in np.where(Rp["reasons"][t][b])[0]) for b in range(B)] for t in range(K)]
for t in range(K):
    for b in range(B):
        kk = SEGS["test"][0] + t; s = Rp["sym"][b]
        rows_.append({"date": dts[t].date(), "symbol": md.symbols[s], "investor": Rp["inv"][b],
                      "spy_regime": M.REGIME_NAMES[md.bench_regime[kk]] if md.bench_regime[kk] >= 0 else "unknown",
                      "selected_policy_id": int(Rp["selected"][t, b]), "selected_policy": M.POLICY_NAMES[int(Rp["selected"][t, b])],
                      **{f"p_select_{M.POLICY_KEYS[i]}": float(Rp["probs"][t, b, i]) for i in range(M.N_POLICIES)},
                      "proposed_action": ACTION_NAMES[int(Rp["proposed"][t, b])], "final_action": ACTION_NAMES[int(Rp["final"][t, b])],
                      "entry_rejected": bool(Rp["blocked"][t, b]), "exit_forced": bool(Rp["forced"][t, b]), "gate_reasons": reason_str[t][b],
                      "step_return": float(Rp["ret"][t, b]), "step_reward": float(Rp["reward"][t, b]), "equity": float(Rp["equity"][t, b]), "behaviour_distance": float(Rp["beh"][t, b])})
SELECTION_LOG = pd.DataFrame(rows_)
SELECTION_LOG.to_csv(ART_DIR / "policy_selection_log.csv", index=False)

curves = []
for n in MAIN_METHODS + [x for x in ABL_NAMES if x != NAME_PPO]:
    for a in ARCH_NAMES + ["ALL"]:
        r = portfolio_returns(RUNS["test"][n], a)
        curves.append(pd.DataFrame({"date": dates_test, "investor": a, "method": n, "return": r, "equity": np.cumprod(1 + r)}))
for n, r in reference_rows("test").items():
    curves.append(pd.DataFrame({"date": dates_test, "investor": "n/a", "method": n, "return": r, "equity": np.cumprod(1 + r)}))
pd.concat(curves, ignore_index=True).to_csv(ART_DIR / "equity_curve.csv", index=False)

for f in ("meta_rl_core.py", "meta_rl_risk.py"):
    shutil.copy(f, ART_DIR / f)
ARTIFACTS = ["meta_rl_selector.zip", "meta_rl_config.json", "meta_rl_observation_schema.json", "risk_controller_config.json", "evaluation_metrics.json",
             "policy_selection_log.csv", "equity_curve.csv", "meta_rl_core.py", "meta_rl_risk.py"]
for a in ARTIFACTS:
    assert (ART_DIR / a).exists(), f"missing artifact {a}"
    print(f"saved {(ART_DIR / a).resolve()}  ({(ART_DIR / a).stat().st_size / 1024:.1f} KiB)")
shutil.make_archive(str(ART_DIR), "zip", ART_DIR)
print(f"zip bundle: {Path(str(ART_DIR) + '.zip').resolve()}")
try:
    from google.colab import files  # noqa: F401  (only exists in Colab)
    files.download(str(ART_DIR) + ".zip")
except Exception:  # noqa: BLE001
    print("(not running in Colab or download blocked: copy the zip from the path above)")

## 14. Inference demonstration
`load_meta_agent(path)` loads the checkpoint, the sidecar configs, the five frozen policies and the deterministic gate **from files only** (no data download, no training). The three steps are separate calls:

1. `select_policy(observation)` → policy id, policy name and the selection probabilities (PPO only);
2. the chosen frozen policy's public `predict(state)` → proposed action and confidence;
3. the deterministic gate → final action, risk decision and rejection reason.

`predict_action(observation)` chains the three and returns all of it; it **never executes** anything. Simulated execution is a separate component (`AccountBatch` / `SleeveSim`). The `observation` is a dictionary: `vector` (the 108-dim selector input), `policy_inputs` (five states in each policy's public format), `risk_context` (raw risk features, account, investor limits) and `meta`.

In [ ]:
agent = M.load_meta_agent(ART_DIR / "meta_rl_selector.zip", policy_dir=POLICY_ROOT, device="cpu")
print("loaded:", type(agent).__name__, "| policies:", [p["name"] for p in meta_config["policies"]])

k_demo = SEGS["test"][0] + 40; s_demo = 0
sim = M.SleeveSim(md, bank, GATE, ECFG, [s_demo], [ARCH["balanced"]], k_demo)
sim.compute_candidates()
observation = sim.export_observation(0, SHADOW["test"][k_demo - SEGS["test"][0], [s_demo]])
print("\nobservation keys:", list(observation), "| vector:", observation["vector"].shape, "| symbol/date:", observation["meta"])

sel = M.select_policy(observation)
print("\n1) select_policy ->", sel["policy_id"], sel["policy_name"])
print("   probabilities:", {k: round(v, 3) for k, v in sel["selection_probabilities"].items()})
prop = agent.propose(sel["policy_id"], observation)
print("2) chosen policy proposes ->", prop["action_name"], f"(confidence {prop['confidence']:.3f})")
verdict = agent.risk_review(prop, observation)
print("3) risk gate ->", verdict["final_action_name"], "| approved:", verdict["approved"], "| reasons:", verdict["reasons"])
out = M.predict_action(observation)
print("\npredict_action ->", {k: out[k] for k in ("final_action_name", "rejection_reason")}, "| selected:", out["selected_policy"]["policy_name"])
assert out["final_action"] == verdict["final_action"]

# the gate overrules the selection: same observation, account deep in drawdown holding a losing position
obs_stress = copy.deepcopy(observation)
rc = obs_stress["risk_context"]; rc["position"] = 1; rc["unrealized_pnl"] = -0.075; rc["holding_bars"] = 9; rc["account"]["drawdown"] = 0.20
obs_stress["policy_inputs"][sel["policy_id"]].update(position=1, unrealized_pnl=-0.075, holding_bars=9)
o2 = M.predict_action(obs_stress)
print("\nstress example (position -7.5%, account 20% below peak): proposal =", o2["proposal"]["action_name"], "-> FINAL =", o2["final_action_name"], "| reasons:", o2["rejection_reason"])
assert o2["final_action"] == EXIT
obs_off = copy.deepcopy(observation); obs_off["risk_context"]["investor"]["authorised"] = False
o3 = M.predict_action(obs_off)
print("kill switch example (trading not authorised): proposal =", o3["proposal"]["action_name"], "-> FINAL =", o3["final_action_name"], "| reasons:", o3["rejection_reason"])
try:
    M.select_policy(np.zeros(10))
except ValueError as exc:
    print("input guard ->", exc)

# the deployed interface agrees with the training/evaluation path
rng_ = np.random.default_rng(0); agree = 0; N_CHECK = 25
for _ in range(N_CHECK):
    kk = int(rng_.integers(SEGS["test"][0], SEGS["test"][1] - 1)); ss = int(rng_.integers(md.S))
    sm = M.SleeveSim(md, bank, GATE, ECFG, [ss], [ARCH["balanced"]], kk); sm.compute_candidates()
    obs_k = sm.export_observation(0, SHADOW["test"][kk - SEGS["test"][0], [ss]])
    chosen = agent.select_policy(obs_k)
    act_fast, _ = sm.proposal_of(np.array([chosen["policy_id"]]))
    act_public = agent.propose(chosen["policy_id"], obs_k)["action"]
    agree += int(int(act_fast[0]) == act_public)
assert agree == N_CHECK, "deployed (public) and training (fast) paths disagree"
print(f"\ndeployed interface vs evaluation path: {agree}/{N_CHECK} identical proposals")

log_path = M.save_selection_log(ART_DIR / "inference_demo_log.csv")
print("\nselection log saved:", log_path, "\n", pd.read_csv(log_path).T.to_string()[:1200])

## 15. Integration report

In [ ]:
print("=" * 104)
print("FINAL INTEGRATION REPORT: Personalized Meta-RL Strategy Selector (research prototype, offline evaluation / paper trading only)")
print("=" * 104)
print("\n1) REQUIRED INPUT ARTIFACTS (each from its own notebook; each in its OWN folder, because file names repeat across zips)")
for p in meta_config["policies"]:
    print(f"  [{p['policy_id']}] {p['name']:34s} {p['artifact']:28s} {p['n_sequence_features']:>2d} sequence cols, {p['n_static_features']:>2d} static cols, seq_len {p['seq_len']}, "
          f"allow_short={p['allow_short']}, sha256 {p['sha256'][:12]}...")
    print(f"        files: {', '.join(p['required_files'])}")
print("  Expected schema contract (verified at load time):")
print("    - action mapping 0 HOLD, 1 BUY, 2 SELL, 3 EXIT; decision at the close of bar t, fill at the open of t+1")
print("    - identical first 30 sequence features; static: SP-01 6 cols (prefix of) SP-02..04 8 cols (prefix of) SP-05 12 cols")
print("    - the same policy train/validation split dates and benchmark; frozen weights (never updated here)")
print(f"\n2) OBSERVATION SCHEMA: {M.OBS_DIM} float32 dims in [-10, 10]: " + ", ".join(f"{g.split('_')[0]} {b - a}" for g, (a, b) in groups.items()) + "  (see meta_rl_observation_schema.json)")
print("\n3) OUTPUT INTERFACE (meta_rl_core.py)")
print("    load_meta_agent(path, policy_dir=None, device='cpu') -> MetaAgent")
print("    select_policy(obs)  -> {policy_id, policy_name, policy_key, selection_probabilities}")
print("    predict_action(obs) -> {selected_policy, proposal{action, action_name, confidence, action_probs}, risk_decision{approved, reasons}, final_action, final_action_name, rejection_reason}")
print("    save_selection_log(path) -> CSV of every predict_action call")
print("    obs = {'vector': (108,), 'policy_inputs': [5 policy states], 'risk_context': {...}, 'meta': {...}}")
print("\n4) STEPS TO CONNECT A FUTURE FastAPI BACKEND (illustrative; not executed here)")
steps = [
    "Package: copy meta_rl_selector.zip, meta_rl_config.json, risk_controller_config.json, meta_rl_core.py, meta_rl_risk.py and the five policy folders; pip install torch gymnasium stable-baselines3 pandas numpy.",
    "Start-up: agent = load_meta_agent('meta_rl_selector.zip', policy_dir='<five policy folders>'); verify the SHA-256 of every policy .pt against meta_rl_config.json.",
    "Market data: keep daily OHLCV history (>= 600 bars) for each symbol and the benchmark from YOUR data feed (never fetched inside the model code); build meta_rl_core.MarketData(raw, bench, agent.bank).",
    "Account state: from the paper-trading ledger supply position, entry price (-> unrealised P&L), bars held, account equity/peak (-> drawdown), gross exposure and ~20-bar equity change.",
    "Shadow performance (observation group D2): call run_shadow over a trailing window (for example the last 250 bars) with the reference investor; use its last row.",
    "Observation: SleeveSim(...).export_observation(...) after setting the account state, or reproduce its fields; the vector, the five policy states and the risk context come from the same function used in training.",
    "Endpoint POST /decision: body = symbol, investor profile and limits, account state -> response = predict_action(observation) (selected policy, proposal, risk decision, final action, rejection reason). Log every call with save_selection_log.",
    "Execution: a SEPARATE service executes only approved final actions in the PAPER account at the next open with the same fee/slippage model; no live broker credentials in this process.",
    "Operations: monitor the rejection/override rate, the selection frequencies and the realised-vs-simulated slippage; keep the kill switch (authorised=False) and re-validate on new data before any change.",
]
for i, s in enumerate(steps, 1):
    print(f"    {i}. {s}")
print("""
    # illustrative sketch (not executed)
    # from fastapi import FastAPI; import meta_rl_core as M
    # app = FastAPI(); agent = M.load_meta_agent('meta_rl_selector.zip', policy_dir='policies/')
    # @app.post('/decision')
    # def decision(req: DecisionRequest):
    #     obs = build_observation(req)            # uses MarketData + SleeveSim.export_observation as in section 14
    #     return agent.predict_action(obs)        # selected policy, proposal, risk decision, final action, rejection reason""")
print("\n5) ARTIFACT PATHS")
for a in ARTIFACTS:
    print(f"    {(ART_DIR / a).resolve()}")
print(f"    {Path(str(ART_DIR) + '.zip').resolve()}   (zip of everything above)")
print("\n6) RESULT ON THE HELD-OUT TEST PERIOD (not a prediction):")
print("    " + VERDICT)
print(show_cols(TEST_TABLE["ALL"].loc[[NAME_PPO, NAME_HEUR, NAME_BH, NAME_GC] + POLICY_ALONE]).to_string())
print("\n7) KNOWN LIMITATIONS")
for s in ["Investors are synthetic; nothing about any real person was learned. Real trade history would need a validated behaviour encoder or richer statistics.",
          "The selector's training period is short in the leakage-free `policy_oos` mode; one PPO run per configuration, so differences between variants are within noise unless confirmed with several seeds.",
          "Shadow performance features use a reference investor; the policies' proposals use the actual investor profile.",
          "Single-asset sleeves with fixed-fraction sizing; no cross-asset portfolio optimisation, no leverage, daily bars only, next-open fills with fixed slippage, no market impact or borrow availability.",
          "The test period is one regime mix and a hand-picked list of liquid survivors; no walk-forward retraining of the policies or the selector was done.",
          "The risk gate cannot prevent overnight gap losses on exposure that is already open.",
          "Reward weights are assumptions; sensitivity was measured for a few variants only."]:
    print("    -", s)
print("\nResearch prototype for offline evaluation and paper trading. Not investment advice; no profitability or capital protection is guaranteed or implied.")
print("=" * 104)